In [21]:
import time
import base64
import pandas as pd
from bs4 import BeautifulSoup
from pathlib import Path

from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait, Select
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager

# ══════════════════════════════════════════════
# ▶ CONFIGURACIÓN
# ══════════════════════════════════════════════

URL            = "https://admision.unmsm.edu.pe/Website20262/A/091/results.html"
BATCH_RUTAS    = []   # ejemplo: ["A/091", "A/092"]
ARCHIVO_SALIDA = "resultados_unmsm.csv"
BASE_URL       = "https://admision.unmsm.edu.pe/Website20262"
WAIT_SEGUNDOS  = 8

# ══════════════════════════════════════════════


def crear_driver():
    opciones = Options()
    opciones.add_argument("--headless=new")
    opciones.add_argument("--no-sandbox")
    opciones.add_argument("--disable-dev-shm-usage")
    opciones.add_argument("--disable-gpu")
    opciones.add_argument("--window-size=1920,1080")
    opciones.add_argument(
        "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36"
    )
    return webdriver.Chrome(
        service=Service(ChromeDriverManager().install()),
        options=opciones
    )


def mostrar_todas_las_filas(driver):
    """
    Intenta mostrar TODOS los registros de una vez usando 3 estrategias:
    1. Cambiar el select de longitud de DataTables a -1 (All) vía Selenium
    2. Forzarlo por JavaScript directamente sobre la instancia DataTable
    3. Si no hay selector, navegar página por página
    """
    wait = WebDriverWait(driver, 10)

    # ── Estrategia 1: select de longitud (típico en DataTables) ──────────
    try:
        select_el = wait.until(
            EC.presence_of_element_located((By.CSS_SELECTOR, "select[name*='length'], select[name*='DataTables']"))
        )
        select = Select(select_el)
        try:
            select.select_by_value("-1")
            print("  [✓] Paginación: seleccionado 'Todos' en el select.")
        except Exception:
            opciones_vals = [o.get_attribute("value") for o in select.options]
            select.select_by_value(opciones_vals[-1])
            print(f"  [✓] Paginación: seleccionado valor máximo ({opciones_vals[-1]}) en el select.")
        time.sleep(3)
        return "select"
    except Exception:
        pass

    # ── Estrategia 2: forzar via JavaScript DataTables API ───────────────
    try:
        driver.execute_script("""
            var tables = $.fn.dataTable ? $.fn.dataTable.tables() : [];
            if (tables.length > 0) {
                $(tables[0]).DataTable().page.len(-1).draw();
            }
        """)
        time.sleep(3)
        print("  [✓] Paginación: forzado via DataTables JS API.")
        return "js"
    except Exception:
        pass

    # ── Estrategia 3: paginación manual (clic en "Siguiente") ────────────
    print("  [!] Usando paginación manual página por página...")
    return "manual"


def obtener_html_completo(driver, url: str) -> str:
    """Carga la página y retorna el HTML con TODOS los registros visibles."""
    driver.get(url)
    time.sleep(WAIT_SEGUNDOS)

    modo = mostrar_todas_las_filas(driver)

    if modo == "manual":
        return obtener_html_paginado(driver)

    return driver.page_source


def obtener_html_paginado(driver) -> str:
    """
    Recorre todas las páginas haciendo clic en 'Siguiente'
    y acumula los <tr> de cada página en una tabla unificada.
    """
    todas_filas_html = []
    pagina = 1

    while True:
        soup  = BeautifulSoup(driver.page_source, "html.parser")
        tabla = soup.find("table")
        if tabla:
            filas = tabla.find_all("tr")[1:]  # sin encabezado
            todas_filas_html.extend([str(f) for f in filas])
            print(f"    Página {pagina}: {len(filas)} filas | total acumulado: {len(todas_filas_html)}")

        try:
            siguiente = driver.find_element(
                By.CSS_SELECTOR,
                "a.paginate_button.next:not(.disabled), button.paginate_button.next:not(.disabled)"
            )
            siguiente.click()
            pagina += 1
            time.sleep(2)
        except Exception:
            print(f"    Fin de paginación en página {pagina}.")
            break

    # Reconstruir HTML con todas las filas
    soup_base  = BeautifulSoup(driver.page_source, "html.parser")
    tabla_base = soup_base.find("table")
    if tabla_base:
        tbody = tabla_base.find("tbody")
        if tbody:
            tbody.clear()
            for fila_html in todas_filas_html:
                tbody.append(BeautifulSoup(fila_html, "html.parser"))
    return str(soup_base)


def decodificar_b64(valor: str) -> str:
    try:
        return base64.b64decode(valor).decode("utf-8").strip()
    except Exception:
        return valor.strip()


def extraer_carrera(soup: BeautifulSoup) -> str:
    items = soup.select("ol li, ul.breadcrumb li")
    if items:
        return items[-1].get_text(strip=True)
    h1 = soup.find("h1")
    return h1.get_text(strip=True) if h1 else "DESCONOCIDA"


def normalizar_obs(val: str) -> str:
    v = val.strip().upper()
    if "ALCANZ" in v and "VACANTE" in v:
        return "ALCANZÓ VACANTE"
    if "AUSENTE" in v:
        return "AUSENTE"
    if "ART" in v:
        return "INHABILITADO (Art. 5)"
    return v if v else "SIN OBSERVACIÓN"


def parse_tabla(html: str, url: str) -> pd.DataFrame:
    soup  = BeautifulSoup(html, "html.parser")
    tabla = soup.find("table")

    if tabla is None:
        print(f"[AVISO] No se encontró tabla en: {url}")
        return pd.DataFrame()

    filas = []
    for tr in tabla.find_all("tr")[1:]:
        celdas = tr.find_all("td")
        if not celdas:
            continue

        codigo  = celdas[0].get_text(strip=True) if len(celdas) > 0 else ""

        nombre = ""
        if len(celdas) > 1:
            span = celdas[1].find("span", class_="obfuscated")
            nombre = decodificar_b64(span["data-auth"]) if (span and span.get("data-auth")) else celdas[1].get_text(strip=True)

        escuela = ""
        if len(celdas) > 2:
            span = celdas[2].find("span", class_="obfuscated")
            escuela = decodificar_b64(span["data-auth"]) if (span and span.get("data-auth")) else celdas[2].get_text(strip=True)

        puntaje = celdas[3].get("data-score", "").strip() if len(celdas) > 3 else ""
        merito  = celdas[4].get("data-merit", "").strip() if len(celdas) > 4 else ""
        obs     = celdas[5].get_text(strip=True)          if len(celdas) > 5 else ""

        filas.append({
            "codigo":            codigo,
            "apellidos_nombres": nombre,
            "escuela":           escuela,
            "puntaje":           puntaje,
            "merito_ep":         merito,
            "observacion":       obs,
        })

    if not filas:
        print(f"[AVISO] Sin filas en: {url}")
        return pd.DataFrame()

    df = pd.DataFrame(filas)
    df["codigo"] = df["codigo"].str.strip()
    df["apellidos_nombres"] = (
        df["apellidos_nombres"].str.strip().str.title()
        .str.replace(r"\s{2,}", " ", regex=True)
    )
    df["puntaje"]    = pd.to_numeric(df["puntaje"],  errors="coerce")
    df["merito_ep"]  = pd.to_numeric(df["merito_ep"], errors="coerce")
    df["observacion"] = df["observacion"].apply(normalizar_obs)
    df["carrera"]    = extraer_carrera(soup)
    df["url_fuente"] = url

    df = df.dropna(how="all").drop_duplicates(subset=["codigo"]).reset_index(drop=True)
    return df


def scrape_url(driver, url: str) -> pd.DataFrame:
    print(f"  Abriendo: {url}")
    html = obtener_html_completo(driver, url)
    df   = parse_tabla(html, url)
    print(f"  → {len(df)} registros extraídos.")
    return df


def guardar_csv(df: pd.DataFrame, ruta: Path):
    df.to_csv(ruta, index=False, encoding="utf-8-sig")
    print(f"\n✅ CSV guardado: {ruta.resolve()}")
    print(f"   {len(df)} filas  ×  {len(df.columns)} columnas")
    print("\n── Observaciones ──")
    print(df["observacion"].value_counts().to_string())
    if df["puntaje"].notna().any():
        print("\n── Estadísticas de puntaje ──")
        print(df["puntaje"].describe().round(3).to_string())


# ══════════════════════════════════════════════
# ▶ EJECUCIÓN
# ══════════════════════════════════════════════

driver = crear_driver()

try:
    if BATCH_RUTAS:
        dfs = []
        for ruta in BATCH_RUTAS:
            url = f"{BASE_URL}/{ruta.strip('/')}/results.html"
            df  = scrape_url(driver, url)
            if not df.empty:
                dfs.append(df)
            time.sleep(1.5)
        resultado = pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()
    else:
        resultado = scrape_url(driver, URL)
finally:
    driver.quit()
    print("[INFO] Navegador cerrado.")

if not resultado.empty:
    guardar_csv(resultado, Path(ARCHIVO_SALIDA))
    display(resultado.head(15))
else:
    print("❌ No se obtuvieron datos.")

  Abriendo: https://admision.unmsm.edu.pe/Website20262/A/091/results.html
  [✓] Paginación: forzado via DataTables JS API.
  → 553 registros extraídos.
[INFO] Navegador cerrado.

✅ CSV guardado: C:\Users\confe\OneDrive\Documentos\ANÁLISIS DE DATOS UNMSM 2026\resultados_unmsm.csv
   553 filas  ×  8 columnas

── Observaciones ──
observacion
SIN OBSERVACIÓN          450
ALCANZÓ VACANTE           98
AUSENTE                    3
INHABILITADO (Art. 5)      2

── Estadísticas de puntaje ──
count     550.000
mean      810.595
std       185.217
min       353.000
25%       673.375
50%       812.875
75%       929.312
max      1380.125


,codigo,apellidos_nombres,escuela,puntaje,merito_ep,observacion,carrera,url_fuente
0,656550,"Abad Salgado, Sofia Estrella",ADMINISTRACIÓN,461.375,NaN,SIN OBSERVACIÓN,ADMINISTRACIÓN,https://admision.unmsm.edu.pe/Website20262/A/0...
1,565808,"Acuña Echebarria, Yamila Soraya",ADMINISTRACIÓN,1074.875,41.0,ALCANZÓ VACANTE,ADMINISTRACIÓN,https://admision.unmsm.edu.pe/Website20262/A/0...
2,658780,"Acuña Pacotaipe, Dulce Maria",ADMINISTRACIÓN,797.375,NaN,SIN OBSERVACIÓN,ADMINISTRACIÓN,https://admision.unmsm.edu.pe/Website20262/A/0...
3,651014,"Aguero Vidarte, Adrian Sebastian",ADMINISTRACIÓN,826.125,NaN,INHABILITADO (Art. 5),ADMINISTRACIÓN,https://admision.unmsm.edu.pe/Website20262/A/0...
4,658610,"Aguirre Ramos, Jogan Alyair",ADMINISTRACIÓN,780.125,NaN,SIN OBSERVACIÓN,ADMINISTRACIÓN,https://admision.unmsm.edu.pe/Website20262/A/0...
5,647488,"Agurto Linche, Lesly Virginia",ADMINISTRACIÓN,589.750,NaN,SIN OBSERVACIÓN,ADMINISTRACIÓN,https://admision.unmsm.edu.pe/Website20262/A/0...
6,550314,"Alania Vila, Nicolas Josh",ADMINISTRACIÓN,869.500,NaN,SIN OBSERVACIÓN,ADMINISTRACIÓN,https://admision.unmsm.edu.pe/Website20262/A/0...
7,654390,"Alarcon Romero, Estefano",ADMINISTRACIÓN,987.125,NaN,SIN OBSERVACIÓN,ADMINISTRACIÓN,https://admision.unmsm.edu.pe/Website20262/A/0...
8,651804,"Albino Ramirez, Maricielo Cynthia",ADMINISTRACIÓN,809.500,NaN,SIN OBSERVACIÓN,ADMINISTRACIÓN,https://admision.unmsm.edu.pe/Website20262/A/0...
9,659986,"Albites Huamani, Estrella Alexia",ADMINISTRACIÓN,663.375,NaN,SIN OBSERVACIÓN,ADMINISTRACIÓN,https://admision.unmsm.edu.pe/Website20262/A/0...


In [22]:
import pandas as pd

df = pd.read_csv("resultados_unmsm.csv")

vacantes = df[df["observacion"] == "ALCANZÓ VACANTE"]

max_p = vacantes["puntaje"].max()
min_p = vacantes["puntaje"].min()

print(f"Carrera: {df['carrera'].iloc[0]}")
print(f"Vacantes obtenidas: {len(vacantes)}")
print(f"Puntaje máximo: {max_p}")
print(f"Puntaje mínimo: {min_p}")

display(vacantes[["codigo","apellidos_nombres","puntaje","merito_ep"]]
        .sort_values("puntaje", ascending=False)
        .reset_index(drop=True))

Carrera: ADMINISTRACIÓN
Vacantes obtenidas: 98
Puntaje máximo: 1380.125
Puntaje mínimo: 999.125


,codigo,apellidos_nombres,puntaje,merito_ep
0,551459,"Carrasco Rodriguez, Anthony Bill",1380.125,1.0
1,657468,"Rivas Fuentes, Daniel Adrian",1378.625,2.0
2,651318,"Camiloaga Varas, Nilton Fabrizzio",1341.125,3.0
3,589125,"Litano Suero, Bryan Carlos Alberto",1282.875,4.0
4,652274,"Sequeiros Olivera, Marvia Alexandra",1254.250,5.0
...,...,...,...,...
93,645516,"Perez Valverde, Jordan Enrique",1003.125,94.0
94,655501,"Huaman Pachas, Dayla Shanthal",1001.875,95.0
95,556499,"Mazuelos Carpio, Joaquin Florencio",1000.250,96.0
96,645715,"Espinoza Pari, Josh Angelo",999.125,97.0


In [23]:
import time
import base64
import pandas as pd
from bs4 import BeautifulSoup
from pathlib import Path

from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait, Select
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager

# ══════════════════════════════════════════════
# ▶ CONFIGURACIÓN
# ══════════════════════════════════════════════

URL            = "https://admision.unmsm.edu.pe/Website20262/A/094/results.html"
BATCH_RUTAS    = []   # ejemplo: ["A/091", "A/092"]
ARCHIVO_SALIDA = "resultados_unmsm_gastronomia.csv"
BASE_URL       = "https://admision.unmsm.edu.pe/Website20262"
WAIT_SEGUNDOS  = 8

# ══════════════════════════════════════════════


def crear_driver():
    opciones = Options()
    opciones.add_argument("--headless=new")
    opciones.add_argument("--no-sandbox")
    opciones.add_argument("--disable-dev-shm-usage")
    opciones.add_argument("--disable-gpu")
    opciones.add_argument("--window-size=1920,1080")
    opciones.add_argument(
        "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36"
    )
    return webdriver.Chrome(
        service=Service(ChromeDriverManager().install()),
        options=opciones
    )


def mostrar_todas_las_filas(driver):
    """
    Intenta mostrar TODOS los registros de una vez usando 3 estrategias:
    1. Cambiar el select de longitud de DataTables a -1 (All) vía Selenium
    2. Forzarlo por JavaScript directamente sobre la instancia DataTable
    3. Si no hay selector, navegar página por página
    """
    wait = WebDriverWait(driver, 10)

    # ── Estrategia 1: select de longitud (típico en DataTables) ──────────
    try:
        select_el = wait.until(
            EC.presence_of_element_located((By.CSS_SELECTOR, "select[name*='length'], select[name*='DataTables']"))
        )
        select = Select(select_el)
        try:
            select.select_by_value("-1")
            print("  [✓] Paginación: seleccionado 'Todos' en el select.")
        except Exception:
            opciones_vals = [o.get_attribute("value") for o in select.options]
            select.select_by_value(opciones_vals[-1])
            print(f"  [✓] Paginación: seleccionado valor máximo ({opciones_vals[-1]}) en el select.")
        time.sleep(3)
        return "select"
    except Exception:
        pass

    # ── Estrategia 2: forzar via JavaScript DataTables API ───────────────
    try:
        driver.execute_script("""
            var tables = $.fn.dataTable ? $.fn.dataTable.tables() : [];
            if (tables.length > 0) {
                $(tables[0]).DataTable().page.len(-1).draw();
            }
        """)
        time.sleep(3)
        print("  [✓] Paginación: forzado via DataTables JS API.")
        return "js"
    except Exception:
        pass

    # ── Estrategia 3: paginación manual (clic en "Siguiente") ────────────
    print("  [!] Usando paginación manual página por página...")
    return "manual"


def obtener_html_completo(driver, url: str) -> str:
    """Carga la página y retorna el HTML con TODOS los registros visibles."""
    driver.get(url)
    time.sleep(WAIT_SEGUNDOS)

    modo = mostrar_todas_las_filas(driver)

    if modo == "manual":
        return obtener_html_paginado(driver)

    return driver.page_source


def obtener_html_paginado(driver) -> str:
    """
    Recorre todas las páginas haciendo clic en 'Siguiente'
    y acumula los <tr> de cada página en una tabla unificada.
    """
    todas_filas_html = []
    pagina = 1

    while True:
        soup  = BeautifulSoup(driver.page_source, "html.parser")
        tabla = soup.find("table")
        if tabla:
            filas = tabla.find_all("tr")[1:]  # sin encabezado
            todas_filas_html.extend([str(f) for f in filas])
            print(f"    Página {pagina}: {len(filas)} filas | total acumulado: {len(todas_filas_html)}")

        try:
            siguiente = driver.find_element(
                By.CSS_SELECTOR,
                "a.paginate_button.next:not(.disabled), button.paginate_button.next:not(.disabled)"
            )
            siguiente.click()
            pagina += 1
            time.sleep(2)
        except Exception:
            print(f"    Fin de paginación en página {pagina}.")
            break

    # Reconstruir HTML con todas las filas
    soup_base  = BeautifulSoup(driver.page_source, "html.parser")
    tabla_base = soup_base.find("table")
    if tabla_base:
        tbody = tabla_base.find("tbody")
        if tbody:
            tbody.clear()
            for fila_html in todas_filas_html:
                tbody.append(BeautifulSoup(fila_html, "html.parser"))
    return str(soup_base)


def decodificar_b64(valor: str) -> str:
    try:
        return base64.b64decode(valor).decode("utf-8").strip()
    except Exception:
        return valor.strip()


def extraer_carrera(soup: BeautifulSoup) -> str:
    items = soup.select("ol li, ul.breadcrumb li")
    if items:
        return items[-1].get_text(strip=True)
    h1 = soup.find("h1")
    return h1.get_text(strip=True) if h1 else "DESCONOCIDA"


def normalizar_obs(val: str) -> str:
    v = val.strip().upper()
    if "ALCANZ" in v and "VACANTE" in v:
        return "ALCANZÓ VACANTE"
    if "AUSENTE" in v:
        return "AUSENTE"
    if "ART" in v:
        return "INHABILITADO (Art. 5)"
    return v if v else "SIN OBSERVACIÓN"


def parse_tabla(html: str, url: str) -> pd.DataFrame:
    soup  = BeautifulSoup(html, "html.parser")
    tabla = soup.find("table")

    if tabla is None:
        print(f"[AVISO] No se encontró tabla en: {url}")
        return pd.DataFrame()

    filas = []
    for tr in tabla.find_all("tr")[1:]:
        celdas = tr.find_all("td")
        if not celdas:
            continue

        codigo  = celdas[0].get_text(strip=True) if len(celdas) > 0 else ""

        nombre = ""
        if len(celdas) > 1:
            span = celdas[1].find("span", class_="obfuscated")
            nombre = decodificar_b64(span["data-auth"]) if (span and span.get("data-auth")) else celdas[1].get_text(strip=True)

        escuela = ""
        if len(celdas) > 2:
            span = celdas[2].find("span", class_="obfuscated")
            escuela = decodificar_b64(span["data-auth"]) if (span and span.get("data-auth")) else celdas[2].get_text(strip=True)

        puntaje = celdas[3].get("data-score", "").strip() if len(celdas) > 3 else ""
        merito  = celdas[4].get("data-merit", "").strip() if len(celdas) > 4 else ""
        obs     = celdas[5].get_text(strip=True)          if len(celdas) > 5 else ""

        filas.append({
            "codigo":            codigo,
            "apellidos_nombres": nombre,
            "escuela":           escuela,
            "puntaje":           puntaje,
            "merito_ep":         merito,
            "observacion":       obs,
        })

    if not filas:
        print(f"[AVISO] Sin filas en: {url}")
        return pd.DataFrame()

    df = pd.DataFrame(filas)
    df["codigo"] = df["codigo"].str.strip()
    df["apellidos_nombres"] = (
        df["apellidos_nombres"].str.strip().str.title()
        .str.replace(r"\s{2,}", " ", regex=True)
    )
    df["puntaje"]    = pd.to_numeric(df["puntaje"],  errors="coerce")
    df["merito_ep"]  = pd.to_numeric(df["merito_ep"], errors="coerce")
    df["observacion"] = df["observacion"].apply(normalizar_obs)
    df["carrera"]    = extraer_carrera(soup)
    df["url_fuente"] = url

    df = df.dropna(how="all").drop_duplicates(subset=["codigo"]).reset_index(drop=True)
    return df


def scrape_url(driver, url: str) -> pd.DataFrame:
    print(f"  Abriendo: {url}")
    html = obtener_html_completo(driver, url)
    df   = parse_tabla(html, url)
    print(f"  → {len(df)} registros extraídos.")
    return df


def guardar_csv(df: pd.DataFrame, ruta: Path):
    df.to_csv(ruta, index=False, encoding="utf-8-sig")
    print(f"\n✅ CSV guardado: {ruta.resolve()}")
    print(f"   {len(df)} filas  ×  {len(df.columns)} columnas")
    print("\n── Observaciones ──")
    print(df["observacion"].value_counts().to_string())
    if df["puntaje"].notna().any():
        print("\n── Estadísticas de puntaje ──")
        print(df["puntaje"].describe().round(3).to_string())


# ══════════════════════════════════════════════
# ▶ EJECUCIÓN
# ══════════════════════════════════════════════

driver = crear_driver()

try:
    if BATCH_RUTAS:
        dfs = []
        for ruta in BATCH_RUTAS:
            url = f"{BASE_URL}/{ruta.strip('/')}/results.html"
            df  = scrape_url(driver, url)
            if not df.empty:
                dfs.append(df)
            time.sleep(1.5)
        resultado = pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()
    else:
        resultado = scrape_url(driver, URL)
finally:
    driver.quit()
    print("[INFO] Navegador cerrado.")

if not resultado.empty:
    guardar_csv(resultado, Path(ARCHIVO_SALIDA))
    display(resultado.head(15))
else:
    print("❌ No se obtuvieron datos.")

  Abriendo: https://admision.unmsm.edu.pe/Website20262/A/094/results.html
  [✓] Paginación: forzado via DataTables JS API.
  → 127 registros extraídos.
[INFO] Navegador cerrado.

✅ CSV guardado: C:\Users\confe\OneDrive\Documentos\ANÁLISIS DE DATOS UNMSM 2026\resultados_unmsm_gastronomia.csv
   127 filas  ×  8 columnas

── Observaciones ──
observacion
SIN OBSERVACIÓN          105
ALCANZÓ VACANTE           20
INHABILITADO (Art. 5)      1
AUSENTE                    1

── Estadísticas de puntaje ──
count     126.000
mean      783.453
std       184.765
min       401.875
25%       648.031
50%       789.000
75%       904.719
max      1325.125


,codigo,apellidos_nombres,escuela,puntaje,merito_ep,observacion,carrera,url_fuente
0,654480,"Acosta Bravo, Paul Alberto",ADMINISTRACIÓN DE LA GASTRONOMÍA,988.250,20.0,ALCANZÓ VACANTE,ADMINISTRACIÓN DE LA GASTRONOMÍA,https://admision.unmsm.edu.pe/Website20262/A/0...
1,651233,"Agreda Aspajo, Sheyla Jaqueline",ADMINISTRACIÓN DE LA GASTRONOMÍA,505.250,NaN,SIN OBSERVACIÓN,ADMINISTRACIÓN DE LA GASTRONOMÍA,https://admision.unmsm.edu.pe/Website20262/A/0...
2,593875,"Alejandria Lozano, Jorge Luis",ADMINISTRACIÓN DE LA GASTRONOMÍA,480.125,NaN,SIN OBSERVACIÓN,ADMINISTRACIÓN DE LA GASTRONOMÍA,https://admision.unmsm.edu.pe/Website20262/A/0...
3,650933,"Alva Campos, Rodrigo Alonso",ADMINISTRACIÓN DE LA GASTRONOMÍA,907.125,NaN,SIN OBSERVACIÓN,ADMINISTRACIÓN DE LA GASTRONOMÍA,https://admision.unmsm.edu.pe/Website20262/A/0...
4,578232,"Alva Martinez, Matías Alessandre",ADMINISTRACIÓN DE LA GASTRONOMÍA,1059.625,9.0,ALCANZÓ VACANTE,ADMINISTRACIÓN DE LA GASTRONOMÍA,https://admision.unmsm.edu.pe/Website20262/A/0...
5,654738,"Alvarez Cornejo, Rodrigo Sebastian",ADMINISTRACIÓN DE LA GASTRONOMÍA,686.250,NaN,SIN OBSERVACIÓN,ADMINISTRACIÓN DE LA GASTRONOMÍA,https://admision.unmsm.edu.pe/Website20262/A/0...
6,587050,"Anchante Ortiz, Gonzalo Francisco Yonel",ADMINISTRACIÓN DE LA GASTRONOMÍA,949.375,NaN,SIN OBSERVACIÓN,ADMINISTRACIÓN DE LA GASTRONOMÍA,https://admision.unmsm.edu.pe/Website20262/A/0...
7,553364,"Angeles Toribio, Camila Lorena",ADMINISTRACIÓN DE LA GASTRONOMÍA,817.000,NaN,SIN OBSERVACIÓN,ADMINISTRACIÓN DE LA GASTRONOMÍA,https://admision.unmsm.edu.pe/Website20262/A/0...
8,645968,"Aquije Pillaca, Aliana Paula",ADMINISTRACIÓN DE LA GASTRONOMÍA,996.250,17.0,ALCANZÓ VACANTE,ADMINISTRACIÓN DE LA GASTRONOMÍA,https://admision.unmsm.edu.pe/Website20262/A/0...
9,579406,"Aquino Clavijo, Lucero Jimena",ADMINISTRACIÓN DE LA GASTRONOMÍA,462.875,NaN,SIN OBSERVACIÓN,ADMINISTRACIÓN DE LA GASTRONOMÍA,https://admision.unmsm.edu.pe/Website20262/A/0...


In [24]:
import pandas as pd

df = pd.read_csv("resultados_unmsm_gastronomia.csv")

vacantes = df[df["observacion"] == "ALCANZÓ VACANTE"]

max_p = vacantes["puntaje"].max()
min_p = vacantes["puntaje"].min()

print(f"Carrera: {df['carrera'].iloc[0]}")
print(f"Vacantes obtenidas: {len(vacantes)}")
print(f"Puntaje máximo: {max_p}")
print(f"Puntaje mínimo: {min_p}")

display(vacantes[["codigo","apellidos_nombres","puntaje","merito_ep"]]
        .sort_values("puntaje", ascending=False)
        .reset_index(drop=True))

Carrera: ADMINISTRACIÓN DE LA GASTRONOMÍA
Vacantes obtenidas: 20
Puntaje máximo: 1325.125
Puntaje mínimo: 988.25


,codigo,apellidos_nombres,puntaje,merito_ep
0,651207,"Tumialan Ventura, Carlos Elias",1325.125,1.0
1,658705,"Mejia Rodriguez, Henry Kalev",1147.000,2.0
2,552359,"Rodriguez Lara, Fabrizzio Alessandro",1147.000,3.0
3,651392,"More Chapa, Giovani Mathias",1139.000,4.0
4,650872,"Rodriguez Palomino, Samantha",1083.625,5.0
5,587904,"Ricaldi Chagua, Steve James",1080.750,6.0
6,583850,"Garcia Godos Villaverde, Leonardo",1074.875,7.0
7,654938,"Contreras Arias, Giovanny Gianfranco",1063.625,8.0
8,578232,"Alva Martinez, Matías Alessandre",1059.625,9.0
9,560319,"Castro Alayo, Sebastian Abraham",1042.500,10.0


In [25]:
import time
import base64
import pandas as pd
from bs4 import BeautifulSoup
from pathlib import Path

from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait, Select
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager

# ══════════════════════════════════════════════
# ▶ CONFIGURACIÓN
# ══════════════════════════════════════════════

URL            = "https://admision.unmsm.edu.pe/Website20262/A/093/results.html"
BATCH_RUTAS    = []   # ejemplo: ["A/091", "A/092"]
ARCHIVO_SALIDA = "resultados_unmsm_internacionales.csv"
BASE_URL       = "https://admision.unmsm.edu.pe/Website20262"
WAIT_SEGUNDOS  = 8

# ══════════════════════════════════════════════


def crear_driver():
    opciones = Options()
    opciones.add_argument("--headless=new")
    opciones.add_argument("--no-sandbox")
    opciones.add_argument("--disable-dev-shm-usage")
    opciones.add_argument("--disable-gpu")
    opciones.add_argument("--window-size=1920,1080")
    opciones.add_argument(
        "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36"
    )
    return webdriver.Chrome(
        service=Service(ChromeDriverManager().install()),
        options=opciones
    )


def mostrar_todas_las_filas(driver):
    """
    Intenta mostrar TODOS los registros de una vez usando 3 estrategias:
    1. Cambiar el select de longitud de DataTables a -1 (All) vía Selenium
    2. Forzarlo por JavaScript directamente sobre la instancia DataTable
    3. Si no hay selector, navegar página por página
    """
    wait = WebDriverWait(driver, 10)

    # ── Estrategia 1: select de longitud (típico en DataTables) ──────────
    try:
        select_el = wait.until(
            EC.presence_of_element_located((By.CSS_SELECTOR, "select[name*='length'], select[name*='DataTables']"))
        )
        select = Select(select_el)
        try:
            select.select_by_value("-1")
            print("  [✓] Paginación: seleccionado 'Todos' en el select.")
        except Exception:
            opciones_vals = [o.get_attribute("value") for o in select.options]
            select.select_by_value(opciones_vals[-1])
            print(f"  [✓] Paginación: seleccionado valor máximo ({opciones_vals[-1]}) en el select.")
        time.sleep(3)
        return "select"
    except Exception:
        pass

    # ── Estrategia 2: forzar via JavaScript DataTables API ───────────────
    try:
        driver.execute_script("""
            var tables = $.fn.dataTable ? $.fn.dataTable.tables() : [];
            if (tables.length > 0) {
                $(tables[0]).DataTable().page.len(-1).draw();
            }
        """)
        time.sleep(3)
        print("  [✓] Paginación: forzado via DataTables JS API.")
        return "js"
    except Exception:
        pass

    # ── Estrategia 3: paginación manual (clic en "Siguiente") ────────────
    print("  [!] Usando paginación manual página por página...")
    return "manual"


def obtener_html_completo(driver, url: str) -> str:
    """Carga la página y retorna el HTML con TODOS los registros visibles."""
    driver.get(url)
    time.sleep(WAIT_SEGUNDOS)

    modo = mostrar_todas_las_filas(driver)

    if modo == "manual":
        return obtener_html_paginado(driver)

    return driver.page_source


def obtener_html_paginado(driver) -> str:
    """
    Recorre todas las páginas haciendo clic en 'Siguiente'
    y acumula los <tr> de cada página en una tabla unificada.
    """
    todas_filas_html = []
    pagina = 1

    while True:
        soup  = BeautifulSoup(driver.page_source, "html.parser")
        tabla = soup.find("table")
        if tabla:
            filas = tabla.find_all("tr")[1:]  # sin encabezado
            todas_filas_html.extend([str(f) for f in filas])
            print(f"    Página {pagina}: {len(filas)} filas | total acumulado: {len(todas_filas_html)}")

        try:
            siguiente = driver.find_element(
                By.CSS_SELECTOR,
                "a.paginate_button.next:not(.disabled), button.paginate_button.next:not(.disabled)"
            )
            siguiente.click()
            pagina += 1
            time.sleep(2)
        except Exception:
            print(f"    Fin de paginación en página {pagina}.")
            break

    # Reconstruir HTML con todas las filas
    soup_base  = BeautifulSoup(driver.page_source, "html.parser")
    tabla_base = soup_base.find("table")
    if tabla_base:
        tbody = tabla_base.find("tbody")
        if tbody:
            tbody.clear()
            for fila_html in todas_filas_html:
                tbody.append(BeautifulSoup(fila_html, "html.parser"))
    return str(soup_base)


def decodificar_b64(valor: str) -> str:
    try:
        return base64.b64decode(valor).decode("utf-8").strip()
    except Exception:
        return valor.strip()


def extraer_carrera(soup: BeautifulSoup) -> str:
    items = soup.select("ol li, ul.breadcrumb li")
    if items:
        return items[-1].get_text(strip=True)
    h1 = soup.find("h1")
    return h1.get_text(strip=True) if h1 else "DESCONOCIDA"


def normalizar_obs(val: str) -> str:
    v = val.strip().upper()
    if "ALCANZ" in v and "VACANTE" in v:
        return "ALCANZÓ VACANTE"
    if "AUSENTE" in v:
        return "AUSENTE"
    if "ART" in v:
        return "INHABILITADO (Art. 5)"
    return v if v else "SIN OBSERVACIÓN"


def parse_tabla(html: str, url: str) -> pd.DataFrame:
    soup  = BeautifulSoup(html, "html.parser")
    tabla = soup.find("table")

    if tabla is None:
        print(f"[AVISO] No se encontró tabla en: {url}")
        return pd.DataFrame()

    filas = []
    for tr in tabla.find_all("tr")[1:]:
        celdas = tr.find_all("td")
        if not celdas:
            continue

        codigo  = celdas[0].get_text(strip=True) if len(celdas) > 0 else ""

        nombre = ""
        if len(celdas) > 1:
            span = celdas[1].find("span", class_="obfuscated")
            nombre = decodificar_b64(span["data-auth"]) if (span and span.get("data-auth")) else celdas[1].get_text(strip=True)

        escuela = ""
        if len(celdas) > 2:
            span = celdas[2].find("span", class_="obfuscated")
            escuela = decodificar_b64(span["data-auth"]) if (span and span.get("data-auth")) else celdas[2].get_text(strip=True)

        puntaje = celdas[3].get("data-score", "").strip() if len(celdas) > 3 else ""
        merito  = celdas[4].get("data-merit", "").strip() if len(celdas) > 4 else ""
        obs     = celdas[5].get_text(strip=True)          if len(celdas) > 5 else ""

        filas.append({
            "codigo":            codigo,
            "apellidos_nombres": nombre,
            "escuela":           escuela,
            "puntaje":           puntaje,
            "merito_ep":         merito,
            "observacion":       obs,
        })

    if not filas:
        print(f"[AVISO] Sin filas en: {url}")
        return pd.DataFrame()

    df = pd.DataFrame(filas)
    df["codigo"] = df["codigo"].str.strip()
    df["apellidos_nombres"] = (
        df["apellidos_nombres"].str.strip().str.title()
        .str.replace(r"\s{2,}", " ", regex=True)
    )
    df["puntaje"]    = pd.to_numeric(df["puntaje"],  errors="coerce")
    df["merito_ep"]  = pd.to_numeric(df["merito_ep"], errors="coerce")
    df["observacion"] = df["observacion"].apply(normalizar_obs)
    df["carrera"]    = extraer_carrera(soup)
    df["url_fuente"] = url

    df = df.dropna(how="all").drop_duplicates(subset=["codigo"]).reset_index(drop=True)
    return df


def scrape_url(driver, url: str) -> pd.DataFrame:
    print(f"  Abriendo: {url}")
    html = obtener_html_completo(driver, url)
    df   = parse_tabla(html, url)
    print(f"  → {len(df)} registros extraídos.")
    return df


def guardar_csv(df: pd.DataFrame, ruta: Path):
    df.to_csv(ruta, index=False, encoding="utf-8-sig")
    print(f"\n✅ CSV guardado: {ruta.resolve()}")
    print(f"   {len(df)} filas  ×  {len(df.columns)} columnas")
    print("\n── Observaciones ──")
    print(df["observacion"].value_counts().to_string())
    if df["puntaje"].notna().any():
        print("\n── Estadísticas de puntaje ──")
        print(df["puntaje"].describe().round(3).to_string())


# ══════════════════════════════════════════════
# ▶ EJECUCIÓN
# ══════════════════════════════════════════════

driver = crear_driver()

try:
    if BATCH_RUTAS:
        dfs = []
        for ruta in BATCH_RUTAS:
            url = f"{BASE_URL}/{ruta.strip('/')}/results.html"
            df  = scrape_url(driver, url)
            if not df.empty:
                dfs.append(df)
            time.sleep(1.5)
        resultado = pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()
    else:
        resultado = scrape_url(driver, URL)
finally:
    driver.quit()
    print("[INFO] Navegador cerrado.")

if not resultado.empty:
    guardar_csv(resultado, Path(ARCHIVO_SALIDA))
    display(resultado.head(15))
else:
    print("❌ No se obtuvieron datos.")

  Abriendo: https://admision.unmsm.edu.pe/Website20262/A/093/results.html
  [✓] Paginación: forzado via DataTables JS API.
  → 805 registros extraídos.
[INFO] Navegador cerrado.

✅ CSV guardado: C:\Users\confe\OneDrive\Documentos\ANÁLISIS DE DATOS UNMSM 2026\resultados_unmsm_internacionales.csv
   805 filas  ×  8 columnas

── Observaciones ──
observacion
SIN OBSERVACIÓN          724
ALCANZÓ VACANTE           65
INHABILITADO (Art. 5)     11
AUSENTE                    5

── Estadísticas de puntaje ──
count     800.000
mean      807.316
std       181.468
min       347.500
25%       671.312
50%       797.000
75%       930.625
max      1469.000


,codigo,apellidos_nombres,escuela,puntaje,merito_ep,observacion,carrera,url_fuente
0,649927,"Abal Chavez, Lucia Araceli",ADMINISTRACIÓN DE NEGOCIOS INTERNACIONALES,803.750,NaN,SIN OBSERVACIÓN,ADMINISTRACIÓN DE NEGOCIOS INTERNACIONALES,https://admision.unmsm.edu.pe/Website20262/A/0...
1,590170,"Agüero Baldeon, Victor Esteban",ADMINISTRACIÓN DE NEGOCIOS INTERNACIONALES,774.375,NaN,SIN OBSERVACIÓN,ADMINISTRACIÓN DE NEGOCIOS INTERNACIONALES,https://admision.unmsm.edu.pe/Website20262/A/0...
2,646480,"Aguila Pedraza, Yaritza Nayely",ADMINISTRACIÓN DE NEGOCIOS INTERNACIONALES,982.375,NaN,SIN OBSERVACIÓN,ADMINISTRACIÓN DE NEGOCIOS INTERNACIONALES,https://admision.unmsm.edu.pe/Website20262/A/0...
3,648916,"Aguilar Luis, Miguel Ángel",ADMINISTRACIÓN DE NEGOCIOS INTERNACIONALES,890.750,NaN,SIN OBSERVACIÓN,ADMINISTRACIÓN DE NEGOCIOS INTERNACIONALES,https://admision.unmsm.edu.pe/Website20262/A/0...
4,655144,"Aguilar Rodriguez, Tatiana Elena",ADMINISTRACIÓN DE NEGOCIOS INTERNACIONALES,757.875,NaN,SIN OBSERVACIÓN,ADMINISTRACIÓN DE NEGOCIOS INTERNACIONALES,https://admision.unmsm.edu.pe/Website20262/A/0...
5,570272,"Alanya Huaman, Alex Sander",ADMINISTRACIÓN DE NEGOCIOS INTERNACIONALES,680.125,NaN,SIN OBSERVACIÓN,ADMINISTRACIÓN DE NEGOCIOS INTERNACIONALES,https://admision.unmsm.edu.pe/Website20262/A/0...
6,650438,"Alarcón Ramírez, Ariana Antonella",ADMINISTRACIÓN DE NEGOCIOS INTERNACIONALES,1194.375,17.0,ALCANZÓ VACANTE,ADMINISTRACIÓN DE NEGOCIOS INTERNACIONALES,https://admision.unmsm.edu.pe/Website20262/A/0...
7,573788,"Alcca Luna, Maria Fernanda",ADMINISTRACIÓN DE NEGOCIOS INTERNACIONALES,1099.375,51.0,ALCANZÓ VACANTE,ADMINISTRACIÓN DE NEGOCIOS INTERNACIONALES,https://admision.unmsm.edu.pe/Website20262/A/0...
8,558470,"Alegre Monzon, Bruce Wil",ADMINISTRACIÓN DE NEGOCIOS INTERNACIONALES,844.750,NaN,SIN OBSERVACIÓN,ADMINISTRACIÓN DE NEGOCIOS INTERNACIONALES,https://admision.unmsm.edu.pe/Website20262/A/0...
9,650549,"Alejos Haro, Francesco Alexander",ADMINISTRACIÓN DE NEGOCIOS INTERNACIONALES,734.125,NaN,SIN OBSERVACIÓN,ADMINISTRACIÓN DE NEGOCIOS INTERNACIONALES,https://admision.unmsm.edu.pe/Website20262/A/0...


In [26]:
import pandas as pd

df = pd.read_csv("resultados_unmsm_internacionales.csv")

vacantes = df[df["observacion"] == "ALCANZÓ VACANTE"]

max_p = vacantes["puntaje"].max()
min_p = vacantes["puntaje"].min()

print(f"Carrera: {df['carrera'].iloc[0]}")
print(f"Vacantes obtenidas: {len(vacantes)}")
print(f"Puntaje máximo: {max_p}")
print(f"Puntaje mínimo: {min_p}")

display(vacantes[["codigo","apellidos_nombres","puntaje","merito_ep"]]
        .sort_values("puntaje", ascending=False)
        .reset_index(drop=True))

Carrera: ADMINISTRACIÓN DE NEGOCIOS INTERNACIONALES
Vacantes obtenidas: 65
Puntaje máximo: 1469.0
Puntaje mínimo: 1076.125


,codigo,apellidos_nombres,puntaje,merito_ep
0,587448,"Chambergo Cruz, Bianca Mirella",1469.000,1.0
1,564968,"Mauricio Eleuterio, Joaquin Mauricio",1406.125,2.0
2,594302,"Negron Alvis, Diany Araceli",1282.250,3.0
3,556264,"Olano Juarez, Dayanara Jamileth",1269.750,4.0
4,577652,"Carranza Martinez, Yahaira Guadalupe",1244.625,5.0
...,...,...,...,...
60,563521,"Martinez Orosco, Tracy Alejandra",1081.875,61.0
61,652751,"Aparco Morales, Hcosman Eduardo",1080.750,62.0
62,577904,"Osorio Barzola, Yanely Cintia",1079.625,63.0
63,654529,"Quito Aquiño, Marlit Fabiola",1077.750,64.0


In [27]:
import time
import base64
import pandas as pd
from bs4 import BeautifulSoup
from pathlib import Path

from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait, Select
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager

# ══════════════════════════════════════════════
# ▶ CONFIGURACIÓN
# ══════════════════════════════════════════════

URL            = "https://admision.unmsm.edu.pe/Website20262/A/092/results.html"
BATCH_RUTAS    = []   # ejemplo: ["A/091", "A/092"]
ARCHIVO_SALIDA = "resultados_unmsm_turismo.csv"
BASE_URL       = "https://admision.unmsm.edu.pe/Website20262"
WAIT_SEGUNDOS  = 8

# ══════════════════════════════════════════════


def crear_driver():
    opciones = Options()
    opciones.add_argument("--headless=new")
    opciones.add_argument("--no-sandbox")
    opciones.add_argument("--disable-dev-shm-usage")
    opciones.add_argument("--disable-gpu")
    opciones.add_argument("--window-size=1920,1080")
    opciones.add_argument(
        "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36"
    )
    return webdriver.Chrome(
        service=Service(ChromeDriverManager().install()),
        options=opciones
    )


def mostrar_todas_las_filas(driver):
    """
    Intenta mostrar TODOS los registros de una vez usando 3 estrategias:
    1. Cambiar el select de longitud de DataTables a -1 (All) vía Selenium
    2. Forzarlo por JavaScript directamente sobre la instancia DataTable
    3. Si no hay selector, navegar página por página
    """
    wait = WebDriverWait(driver, 10)

    # ── Estrategia 1: select de longitud (típico en DataTables) ──────────
    try:
        select_el = wait.until(
            EC.presence_of_element_located((By.CSS_SELECTOR, "select[name*='length'], select[name*='DataTables']"))
        )
        select = Select(select_el)
        try:
            select.select_by_value("-1")
            print("  [✓] Paginación: seleccionado 'Todos' en el select.")
        except Exception:
            opciones_vals = [o.get_attribute("value") for o in select.options]
            select.select_by_value(opciones_vals[-1])
            print(f"  [✓] Paginación: seleccionado valor máximo ({opciones_vals[-1]}) en el select.")
        time.sleep(3)
        return "select"
    except Exception:
        pass

    # ── Estrategia 2: forzar via JavaScript DataTables API ───────────────
    try:
        driver.execute_script("""
            var tables = $.fn.dataTable ? $.fn.dataTable.tables() : [];
            if (tables.length > 0) {
                $(tables[0]).DataTable().page.len(-1).draw();
            }
        """)
        time.sleep(3)
        print("  [✓] Paginación: forzado via DataTables JS API.")
        return "js"
    except Exception:
        pass

    # ── Estrategia 3: paginación manual (clic en "Siguiente") ────────────
    print("  [!] Usando paginación manual página por página...")
    return "manual"


def obtener_html_completo(driver, url: str) -> str:
    """Carga la página y retorna el HTML con TODOS los registros visibles."""
    driver.get(url)
    time.sleep(WAIT_SEGUNDOS)

    modo = mostrar_todas_las_filas(driver)

    if modo == "manual":
        return obtener_html_paginado(driver)

    return driver.page_source


def obtener_html_paginado(driver) -> str:
    """
    Recorre todas las páginas haciendo clic en 'Siguiente'
    y acumula los <tr> de cada página en una tabla unificada.
    """
    todas_filas_html = []
    pagina = 1

    while True:
        soup  = BeautifulSoup(driver.page_source, "html.parser")
        tabla = soup.find("table")
        if tabla:
            filas = tabla.find_all("tr")[1:]  # sin encabezado
            todas_filas_html.extend([str(f) for f in filas])
            print(f"    Página {pagina}: {len(filas)} filas | total acumulado: {len(todas_filas_html)}")

        try:
            siguiente = driver.find_element(
                By.CSS_SELECTOR,
                "a.paginate_button.next:not(.disabled), button.paginate_button.next:not(.disabled)"
            )
            siguiente.click()
            pagina += 1
            time.sleep(2)
        except Exception:
            print(f"    Fin de paginación en página {pagina}.")
            break

    # Reconstruir HTML con todas las filas
    soup_base  = BeautifulSoup(driver.page_source, "html.parser")
    tabla_base = soup_base.find("table")
    if tabla_base:
        tbody = tabla_base.find("tbody")
        if tbody:
            tbody.clear()
            for fila_html in todas_filas_html:
                tbody.append(BeautifulSoup(fila_html, "html.parser"))
    return str(soup_base)


def decodificar_b64(valor: str) -> str:
    try:
        return base64.b64decode(valor).decode("utf-8").strip()
    except Exception:
        return valor.strip()


def extraer_carrera(soup: BeautifulSoup) -> str:
    items = soup.select("ol li, ul.breadcrumb li")
    if items:
        return items[-1].get_text(strip=True)
    h1 = soup.find("h1")
    return h1.get_text(strip=True) if h1 else "DESCONOCIDA"


def normalizar_obs(val: str) -> str:
    v = val.strip().upper()
    if "ALCANZ" in v and "VACANTE" in v:
        return "ALCANZÓ VACANTE"
    if "AUSENTE" in v:
        return "AUSENTE"
    if "ART" in v:
        return "INHABILITADO (Art. 5)"
    return v if v else "SIN OBSERVACIÓN"


def parse_tabla(html: str, url: str) -> pd.DataFrame:
    soup  = BeautifulSoup(html, "html.parser")
    tabla = soup.find("table")

    if tabla is None:
        print(f"[AVISO] No se encontró tabla en: {url}")
        return pd.DataFrame()

    filas = []
    for tr in tabla.find_all("tr")[1:]:
        celdas = tr.find_all("td")
        if not celdas:
            continue

        codigo  = celdas[0].get_text(strip=True) if len(celdas) > 0 else ""

        nombre = ""
        if len(celdas) > 1:
            span = celdas[1].find("span", class_="obfuscated")
            nombre = decodificar_b64(span["data-auth"]) if (span and span.get("data-auth")) else celdas[1].get_text(strip=True)

        escuela = ""
        if len(celdas) > 2:
            span = celdas[2].find("span", class_="obfuscated")
            escuela = decodificar_b64(span["data-auth"]) if (span and span.get("data-auth")) else celdas[2].get_text(strip=True)

        puntaje = celdas[3].get("data-score", "").strip() if len(celdas) > 3 else ""
        merito  = celdas[4].get("data-merit", "").strip() if len(celdas) > 4 else ""
        obs     = celdas[5].get_text(strip=True)          if len(celdas) > 5 else ""

        filas.append({
            "codigo":            codigo,
            "apellidos_nombres": nombre,
            "escuela":           escuela,
            "puntaje":           puntaje,
            "merito_ep":         merito,
            "observacion":       obs,
        })

    if not filas:
        print(f"[AVISO] Sin filas en: {url}")
        return pd.DataFrame()

    df = pd.DataFrame(filas)
    df["codigo"] = df["codigo"].str.strip()
    df["apellidos_nombres"] = (
        df["apellidos_nombres"].str.strip().str.title()
        .str.replace(r"\s{2,}", " ", regex=True)
    )
    df["puntaje"]    = pd.to_numeric(df["puntaje"],  errors="coerce")
    df["merito_ep"]  = pd.to_numeric(df["merito_ep"], errors="coerce")
    df["observacion"] = df["observacion"].apply(normalizar_obs)
    df["carrera"]    = extraer_carrera(soup)
    df["url_fuente"] = url

    df = df.dropna(how="all").drop_duplicates(subset=["codigo"]).reset_index(drop=True)
    return df


def scrape_url(driver, url: str) -> pd.DataFrame:
    print(f"  Abriendo: {url}")
    html = obtener_html_completo(driver, url)
    df   = parse_tabla(html, url)
    print(f"  → {len(df)} registros extraídos.")
    return df


def guardar_csv(df: pd.DataFrame, ruta: Path):
    df.to_csv(ruta, index=False, encoding="utf-8-sig")
    print(f"\n✅ CSV guardado: {ruta.resolve()}")
    print(f"   {len(df)} filas  ×  {len(df.columns)} columnas")
    print("\n── Observaciones ──")
    print(df["observacion"].value_counts().to_string())
    if df["puntaje"].notna().any():
        print("\n── Estadísticas de puntaje ──")
        print(df["puntaje"].describe().round(3).to_string())


# ══════════════════════════════════════════════
# ▶ EJECUCIÓN
# ══════════════════════════════════════════════

driver = crear_driver()

try:
    if BATCH_RUTAS:
        dfs = []
        for ruta in BATCH_RUTAS:
            url = f"{BASE_URL}/{ruta.strip('/')}/results.html"
            df  = scrape_url(driver, url)
            if not df.empty:
                dfs.append(df)
            time.sleep(1.5)
        resultado = pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()
    else:
        resultado = scrape_url(driver, URL)
finally:
    driver.quit()
    print("[INFO] Navegador cerrado.")

if not resultado.empty:
    guardar_csv(resultado, Path(ARCHIVO_SALIDA))
    display(resultado.head(15))
else:
    print("❌ No se obtuvieron datos.")

  Abriendo: https://admision.unmsm.edu.pe/Website20262/A/092/results.html
  [✓] Paginación: forzado via DataTables JS API.
  → 172 registros extraídos.
[INFO] Navegador cerrado.

✅ CSV guardado: C:\Users\confe\OneDrive\Documentos\ANÁLISIS DE DATOS UNMSM 2026\resultados_unmsm_turismo.csv
   172 filas  ×  8 columnas

── Observaciones ──
observacion
ALCANZÓ VACANTE    99
SIN OBSERVACIÓN    71
AUSENTE             2

── Estadísticas de puntaje ──
count     170.000
mean      769.454
std       171.458
min       328.750
25%       670.219
50%       784.625
75%       876.844
max      1280.375


,codigo,apellidos_nombres,escuela,puntaje,merito_ep,observacion,carrera,url_fuente
0,568108,"Abad Carranza, Diego Leonardo",ADMINISTRACIÓN DE TURISMO,709.000,NaN,SIN OBSERVACIÓN,ADMINISTRACIÓN DE TURISMO,https://admision.unmsm.edu.pe/Website20262/A/0...
1,650129,"Abanto Cunya, Akemy Sayury",ADMINISTRACIÓN DE TURISMO,958.000,21.0,ALCANZÓ VACANTE,ADMINISTRACIÓN DE TURISMO,https://admision.unmsm.edu.pe/Website20262/A/0...
2,656607,"Abanto Leon, Yorsh Erik",ADMINISTRACIÓN DE TURISMO,827.000,63.0,ALCANZÓ VACANTE,ADMINISTRACIÓN DE TURISMO,https://admision.unmsm.edu.pe/Website20262/A/0...
3,588910,"Acosta Rengifo, Diego Alessandro",ADMINISTRACIÓN DE TURISMO,910.625,34.0,ALCANZÓ VACANTE,ADMINISTRACIÓN DE TURISMO,https://admision.unmsm.edu.pe/Website20262/A/0...
4,657170,"Agüero Iñapi, Ciara Luana",ADMINISTRACIÓN DE TURISMO,881.500,41.0,ALCANZÓ VACANTE,ADMINISTRACIÓN DE TURISMO,https://admision.unmsm.edu.pe/Website20262/A/0...
5,589104,"Aguinaga Acuña, Zoe Mayte",ADMINISTRACIÓN DE TURISMO,457.250,NaN,SIN OBSERVACIÓN,ADMINISTRACIÓN DE TURISMO,https://admision.unmsm.edu.pe/Website20262/A/0...
6,651411,"Alegre Reyes, Miqueas Domingo",ADMINISTRACIÓN DE TURISMO,793.250,79.0,ALCANZÓ VACANTE,ADMINISTRACIÓN DE TURISMO,https://admision.unmsm.edu.pe/Website20262/A/0...
7,648527,"Alonso Medina, Jhordy Alexis",ADMINISTRACIÓN DE TURISMO,1009.375,13.0,ALCANZÓ VACANTE,ADMINISTRACIÓN DE TURISMO,https://admision.unmsm.edu.pe/Website20262/A/0...
8,655556,"Alvarado Ruiz, Luz Karina",ADMINISTRACIÓN DE TURISMO,834.000,62.0,ALCANZÓ VACANTE,ADMINISTRACIÓN DE TURISMO,https://admision.unmsm.edu.pe/Website20262/A/0...
9,571505,"Alzamora Huarancca, Salvador Sebastian",ADMINISTRACIÓN DE TURISMO,919.750,31.0,ALCANZÓ VACANTE,ADMINISTRACIÓN DE TURISMO,https://admision.unmsm.edu.pe/Website20262/A/0...


In [28]:
import pandas as pd

df = pd.read_csv("resultados_unmsm_turismo.csv")

vacantes = df[df["observacion"] == "ALCANZÓ VACANTE"]

max_p = vacantes["puntaje"].max()
min_p = vacantes["puntaje"].min()

print(f"Carrera: {df['carrera'].iloc[0]}")
print(f"Vacantes obtenidas: {len(vacantes)}")
print(f"Puntaje máximo: {max_p}")
print(f"Puntaje mínimo: {min_p}")

display(vacantes[["codigo","apellidos_nombres","puntaje","merito_ep"]]
        .sort_values("puntaje", ascending=False)
        .reset_index(drop=True))

Carrera: ADMINISTRACIÓN DE TURISMO
Vacantes obtenidas: 99
Puntaje máximo: 1280.375
Puntaje mínimo: 758.0


,codigo,apellidos_nombres,puntaje,merito_ep
0,658288,"Romero Romero, Stacy Abigail",1280.375,1.0
1,568040,"Garcia Tezen, Thiago Alonso",1194.375,2.0
2,648948,"Perez Flores, Nicolas Melvin",1119.000,3.0
3,651360,"Berrocal Chavez, Rafaela",1088.750,4.0
4,656939,"Pérez Magencio, Madona",1082.875,5.0
...,...,...,...,...
94,655299,"Ordinola Rodríguez, Clara Sofía",766.750,95.0
95,652640,"Villegas Coronel, Romina Mishelle",763.875,96.0
96,551198,"Gonzales Tecco, Asurim",759.875,97.0
97,649831,"Maurera Ortiz, Kareannys De Los Angeles",759.500,98.0


In [29]:
import time
import base64
import pandas as pd
from bs4 import BeautifulSoup
from pathlib import Path

from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait, Select
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager

# ══════════════════════════════════════════════
# ▶ CONFIGURACIÓN
# ══════════════════════════════════════════════

URL            = "https://admision.unmsm.edu.pe/Website20262/A/095/results.html"
BATCH_RUTAS    = []   # ejemplo: ["A/091", "A/092"]
ARCHIVO_SALIDA = "resultados_unmsm_maritima.csv"
BASE_URL       = "https://admision.unmsm.edu.pe/Website20262"
WAIT_SEGUNDOS  = 8

# ══════════════════════════════════════════════


def crear_driver():
    opciones = Options()
    opciones.add_argument("--headless=new")
    opciones.add_argument("--no-sandbox")
    opciones.add_argument("--disable-dev-shm-usage")
    opciones.add_argument("--disable-gpu")
    opciones.add_argument("--window-size=1920,1080")
    opciones.add_argument(
        "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36"
    )
    return webdriver.Chrome(
        service=Service(ChromeDriverManager().install()),
        options=opciones
    )


def mostrar_todas_las_filas(driver):
    """
    Intenta mostrar TODOS los registros de una vez usando 3 estrategias:
    1. Cambiar el select de longitud de DataTables a -1 (All) vía Selenium
    2. Forzarlo por JavaScript directamente sobre la instancia DataTable
    3. Si no hay selector, navegar página por página
    """
    wait = WebDriverWait(driver, 10)

    # ── Estrategia 1: select de longitud (típico en DataTables) ──────────
    try:
        select_el = wait.until(
            EC.presence_of_element_located((By.CSS_SELECTOR, "select[name*='length'], select[name*='DataTables']"))
        )
        select = Select(select_el)
        try:
            select.select_by_value("-1")
            print("  [✓] Paginación: seleccionado 'Todos' en el select.")
        except Exception:
            opciones_vals = [o.get_attribute("value") for o in select.options]
            select.select_by_value(opciones_vals[-1])
            print(f"  [✓] Paginación: seleccionado valor máximo ({opciones_vals[-1]}) en el select.")
        time.sleep(3)
        return "select"
    except Exception:
        pass

    # ── Estrategia 2: forzar via JavaScript DataTables API ───────────────
    try:
        driver.execute_script("""
            var tables = $.fn.dataTable ? $.fn.dataTable.tables() : [];
            if (tables.length > 0) {
                $(tables[0]).DataTable().page.len(-1).draw();
            }
        """)
        time.sleep(3)
        print("  [✓] Paginación: forzado via DataTables JS API.")
        return "js"
    except Exception:
        pass

    # ── Estrategia 3: paginación manual (clic en "Siguiente") ────────────
    print("  [!] Usando paginación manual página por página...")
    return "manual"


def obtener_html_completo(driver, url: str) -> str:
    """Carga la página y retorna el HTML con TODOS los registros visibles."""
    driver.get(url)
    time.sleep(WAIT_SEGUNDOS)

    modo = mostrar_todas_las_filas(driver)

    if modo == "manual":
        return obtener_html_paginado(driver)

    return driver.page_source


def obtener_html_paginado(driver) -> str:
    """
    Recorre todas las páginas haciendo clic en 'Siguiente'
    y acumula los <tr> de cada página en una tabla unificada.
    """
    todas_filas_html = []
    pagina = 1

    while True:
        soup  = BeautifulSoup(driver.page_source, "html.parser")
        tabla = soup.find("table")
        if tabla:
            filas = tabla.find_all("tr")[1:]  # sin encabezado
            todas_filas_html.extend([str(f) for f in filas])
            print(f"    Página {pagina}: {len(filas)} filas | total acumulado: {len(todas_filas_html)}")

        try:
            siguiente = driver.find_element(
                By.CSS_SELECTOR,
                "a.paginate_button.next:not(.disabled), button.paginate_button.next:not(.disabled)"
            )
            siguiente.click()
            pagina += 1
            time.sleep(2)
        except Exception:
            print(f"    Fin de paginación en página {pagina}.")
            break

    # Reconstruir HTML con todas las filas
    soup_base  = BeautifulSoup(driver.page_source, "html.parser")
    tabla_base = soup_base.find("table")
    if tabla_base:
        tbody = tabla_base.find("tbody")
        if tbody:
            tbody.clear()
            for fila_html in todas_filas_html:
                tbody.append(BeautifulSoup(fila_html, "html.parser"))
    return str(soup_base)


def decodificar_b64(valor: str) -> str:
    try:
        return base64.b64decode(valor).decode("utf-8").strip()
    except Exception:
        return valor.strip()


def extraer_carrera(soup: BeautifulSoup) -> str:
    items = soup.select("ol li, ul.breadcrumb li")
    if items:
        return items[-1].get_text(strip=True)
    h1 = soup.find("h1")
    return h1.get_text(strip=True) if h1 else "DESCONOCIDA"


def normalizar_obs(val: str) -> str:
    v = val.strip().upper()
    if "ALCANZ" in v and "VACANTE" in v:
        return "ALCANZÓ VACANTE"
    if "AUSENTE" in v:
        return "AUSENTE"
    if "ART" in v:
        return "INHABILITADO (Art. 5)"
    return v if v else "SIN OBSERVACIÓN"


def parse_tabla(html: str, url: str) -> pd.DataFrame:
    soup  = BeautifulSoup(html, "html.parser")
    tabla = soup.find("table")

    if tabla is None:
        print(f"[AVISO] No se encontró tabla en: {url}")
        return pd.DataFrame()

    filas = []
    for tr in tabla.find_all("tr")[1:]:
        celdas = tr.find_all("td")
        if not celdas:
            continue

        codigo  = celdas[0].get_text(strip=True) if len(celdas) > 0 else ""

        nombre = ""
        if len(celdas) > 1:
            span = celdas[1].find("span", class_="obfuscated")
            nombre = decodificar_b64(span["data-auth"]) if (span and span.get("data-auth")) else celdas[1].get_text(strip=True)

        escuela = ""
        if len(celdas) > 2:
            span = celdas[2].find("span", class_="obfuscated")
            escuela = decodificar_b64(span["data-auth"]) if (span and span.get("data-auth")) else celdas[2].get_text(strip=True)

        puntaje = celdas[3].get("data-score", "").strip() if len(celdas) > 3 else ""
        merito  = celdas[4].get("data-merit", "").strip() if len(celdas) > 4 else ""
        obs     = celdas[5].get_text(strip=True)          if len(celdas) > 5 else ""

        filas.append({
            "codigo":            codigo,
            "apellidos_nombres": nombre,
            "escuela":           escuela,
            "puntaje":           puntaje,
            "merito_ep":         merito,
            "observacion":       obs,
        })

    if not filas:
        print(f"[AVISO] Sin filas en: {url}")
        return pd.DataFrame()

    df = pd.DataFrame(filas)
    df["codigo"] = df["codigo"].str.strip()
    df["apellidos_nombres"] = (
        df["apellidos_nombres"].str.strip().str.title()
        .str.replace(r"\s{2,}", " ", regex=True)
    )
    df["puntaje"]    = pd.to_numeric(df["puntaje"],  errors="coerce")
    df["merito_ep"]  = pd.to_numeric(df["merito_ep"], errors="coerce")
    df["observacion"] = df["observacion"].apply(normalizar_obs)
    df["carrera"]    = extraer_carrera(soup)
    df["url_fuente"] = url

    df = df.dropna(how="all").drop_duplicates(subset=["codigo"]).reset_index(drop=True)
    return df


def scrape_url(driver, url: str) -> pd.DataFrame:
    print(f"  Abriendo: {url}")
    html = obtener_html_completo(driver, url)
    df   = parse_tabla(html, url)
    print(f"  → {len(df)} registros extraídos.")
    return df


def guardar_csv(df: pd.DataFrame, ruta: Path):
    df.to_csv(ruta, index=False, encoding="utf-8-sig")
    print(f"\n✅ CSV guardado: {ruta.resolve()}")
    print(f"   {len(df)} filas  ×  {len(df.columns)} columnas")
    print("\n── Observaciones ──")
    print(df["observacion"].value_counts().to_string())
    if df["puntaje"].notna().any():
        print("\n── Estadísticas de puntaje ──")
        print(df["puntaje"].describe().round(3).to_string())


# ══════════════════════════════════════════════
# ▶ EJECUCIÓN
# ══════════════════════════════════════════════

driver = crear_driver()

try:
    if BATCH_RUTAS:
        dfs = []
        for ruta in BATCH_RUTAS:
            url = f"{BASE_URL}/{ruta.strip('/')}/results.html"
            df  = scrape_url(driver, url)
            if not df.empty:
                dfs.append(df)
            time.sleep(1.5)
        resultado = pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()
    else:
        resultado = scrape_url(driver, URL)
finally:
    driver.quit()
    print("[INFO] Navegador cerrado.")

if not resultado.empty:
    guardar_csv(resultado, Path(ARCHIVO_SALIDA))
    display(resultado.head(15))
else:
    print("❌ No se obtuvieron datos.")

  Abriendo: https://admision.unmsm.edu.pe/Website20262/A/095/results.html
  [✓] Paginación: forzado via DataTables JS API.
  → 140 registros extraídos.
[INFO] Navegador cerrado.

✅ CSV guardado: C:\Users\confe\OneDrive\Documentos\ANÁLISIS DE DATOS UNMSM 2026\resultados_unmsm_maritima.csv
   140 filas  ×  8 columnas

── Observaciones ──
observacion
SIN OBSERVACIÓN          122
ALCANZÓ VACANTE           16
AUSENTE                    1
INHABILITADO (Art. 5)      1

── Estadísticas de puntaje ──
count     139.000
mean      815.953
std       152.083
min       441.875
25%       719.312
50%       805.000
75%       923.625
max      1227.500


,codigo,apellidos_nombres,escuela,puntaje,merito_ep,observacion,carrera,url_fuente
0,648816,"Aberanga Vilca, Ruth Milagros",ADMINISTRACIÓN MARÍTIMA Y PORTUARIA,819.875,NaN,SIN OBSERVACIÓN,ADMINISTRACIÓN MARÍTIMA Y PORTUARIA,https://admision.unmsm.edu.pe/Website20262/A/0...
1,574160,"Aira Obregon, Maira Daniela",ADMINISTRACIÓN MARÍTIMA Y PORTUARIA,696.250,NaN,SIN OBSERVACIÓN,ADMINISTRACIÓN MARÍTIMA Y PORTUARIA,https://admision.unmsm.edu.pe/Website20262/A/0...
2,650632,"Alama Jiménez, Lucero Analy",ADMINISTRACIÓN MARÍTIMA Y PORTUARIA,915.750,NaN,SIN OBSERVACIÓN,ADMINISTRACIÓN MARÍTIMA Y PORTUARIA,https://admision.unmsm.edu.pe/Website20262/A/0...
3,646484,"Alania Pacori, Sandra",ADMINISTRACIÓN MARÍTIMA Y PORTUARIA,NaN,NaN,AUSENTE,ADMINISTRACIÓN MARÍTIMA Y PORTUARIA,https://admision.unmsm.edu.pe/Website20262/A/0...
4,594449,"Alayo Saavedra, Rosa Maricielo",ADMINISTRACIÓN MARÍTIMA Y PORTUARIA,889.750,NaN,SIN OBSERVACIÓN,ADMINISTRACIÓN MARÍTIMA Y PORTUARIA,https://admision.unmsm.edu.pe/Website20262/A/0...
5,659294,"Alcarraz Pulgar, Ariel Matias",ADMINISTRACIÓN MARÍTIMA Y PORTUARIA,962.000,NaN,SIN OBSERVACIÓN,ADMINISTRACIÓN MARÍTIMA Y PORTUARIA,https://admision.unmsm.edu.pe/Website20262/A/0...
6,585651,"Alejandro Inga, Yamile Lucia",ADMINISTRACIÓN MARÍTIMA Y PORTUARIA,563.625,NaN,SIN OBSERVACIÓN,ADMINISTRACIÓN MARÍTIMA Y PORTUARIA,https://admision.unmsm.edu.pe/Website20262/A/0...
7,656042,"Alfaro Espinal, Angely Carmen Rosa",ADMINISTRACIÓN MARÍTIMA Y PORTUARIA,789.000,NaN,SIN OBSERVACIÓN,ADMINISTRACIÓN MARÍTIMA Y PORTUARIA,https://admision.unmsm.edu.pe/Website20262/A/0...
8,562734,"Aliaga Choquehuanca, Sthephania Monsserat",ADMINISTRACIÓN MARÍTIMA Y PORTUARIA,758.375,NaN,SIN OBSERVACIÓN,ADMINISTRACIÓN MARÍTIMA Y PORTUARIA,https://admision.unmsm.edu.pe/Website20262/A/0...
9,584864,"Alvarado Gutierrez, Jhairo Junior",ADMINISTRACIÓN MARÍTIMA Y PORTUARIA,1067.625,8.0,ALCANZÓ VACANTE,ADMINISTRACIÓN MARÍTIMA Y PORTUARIA,https://admision.unmsm.edu.pe/Website20262/A/0...


In [30]:
import pandas as pd

df = pd.read_csv("resultados_unmsm_maritima.csv")

vacantes = df[df["observacion"] == "ALCANZÓ VACANTE"]

max_p = vacantes["puntaje"].max()
min_p = vacantes["puntaje"].min()

print(f"Carrera: {df['carrera'].iloc[0]}")
print(f"Vacantes obtenidas: {len(vacantes)}")
print(f"Puntaje máximo: {max_p}")
print(f"Puntaje mínimo: {min_p}")

display(vacantes[["codigo","apellidos_nombres","puntaje","merito_ep"]]
        .sort_values("puntaje", ascending=False)
        .reset_index(drop=True))

Carrera: ADMINISTRACIÓN MARÍTIMA Y PORTUARIA
Vacantes obtenidas: 16
Puntaje máximo: 1227.5
Puntaje mínimo: 1025.375


,codigo,apellidos_nombres,puntaje,merito_ep
0,658107,"Vega Pacheco, Andrea Gaara Stefania",1227.500,1.0
1,566014,"Laucata Muñoz, Diego Andres",1182.375,2.0
2,655448,"Urbano Dávila, Ander Yoel",1125.875,3.0
3,654210,"Inga Castrejón, Reynaldo Zait",1108.750,4.0
4,550046,"Fernandez Rodriguez, Flavia Nicol",1096.750,5.0
5,574958,"Chavez Gomez, Fabiana Joana",1089.875,6.0
6,657572,"Puelles Cárdenas, Jessica Alessandra",1074.375,7.0
7,584864,"Alvarado Gutierrez, Jhairo Junior",1067.625,8.0
8,654068,"Japura Muñoz, Joaquin",1060.750,9.0
9,593803,"Montalvan Via, Anyela Yasuri",1041.375,10.0


In [31]:
import time
import base64
import pandas as pd
from bs4 import BeautifulSoup
from pathlib import Path

from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait, Select
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager

# ══════════════════════════════════════════════
# ▶ CONFIGURACIÓN
# ══════════════════════════════════════════════

URL            = "https://admision.unmsm.edu.pe/Website20262/A/153/results.html"
BATCH_RUTAS    = []   # ejemplo: ["A/091", "A/092"]
ARCHIVO_SALIDA = "resultados_unmsm_antropología.csv"
BASE_URL       = "https://admision.unmsm.edu.pe/Website20262"
WAIT_SEGUNDOS  = 8

# ══════════════════════════════════════════════


def crear_driver():
    opciones = Options()
    opciones.add_argument("--headless=new")
    opciones.add_argument("--no-sandbox")
    opciones.add_argument("--disable-dev-shm-usage")
    opciones.add_argument("--disable-gpu")
    opciones.add_argument("--window-size=1920,1080")
    opciones.add_argument(
        "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36"
    )
    return webdriver.Chrome(
        service=Service(ChromeDriverManager().install()),
        options=opciones
    )


def mostrar_todas_las_filas(driver):
    """
    Intenta mostrar TODOS los registros de una vez usando 3 estrategias:
    1. Cambiar el select de longitud de DataTables a -1 (All) vía Selenium
    2. Forzarlo por JavaScript directamente sobre la instancia DataTable
    3. Si no hay selector, navegar página por página
    """
    wait = WebDriverWait(driver, 10)

    # ── Estrategia 1: select de longitud (típico en DataTables) ──────────
    try:
        select_el = wait.until(
            EC.presence_of_element_located((By.CSS_SELECTOR, "select[name*='length'], select[name*='DataTables']"))
        )
        select = Select(select_el)
        try:
            select.select_by_value("-1")
            print("  [✓] Paginación: seleccionado 'Todos' en el select.")
        except Exception:
            opciones_vals = [o.get_attribute("value") for o in select.options]
            select.select_by_value(opciones_vals[-1])
            print(f"  [✓] Paginación: seleccionado valor máximo ({opciones_vals[-1]}) en el select.")
        time.sleep(3)
        return "select"
    except Exception:
        pass

    # ── Estrategia 2: forzar via JavaScript DataTables API ───────────────
    try:
        driver.execute_script("""
            var tables = $.fn.dataTable ? $.fn.dataTable.tables() : [];
            if (tables.length > 0) {
                $(tables[0]).DataTable().page.len(-1).draw();
            }
        """)
        time.sleep(3)
        print("  [✓] Paginación: forzado via DataTables JS API.")
        return "js"
    except Exception:
        pass

    # ── Estrategia 3: paginación manual (clic en "Siguiente") ────────────
    print("  [!] Usando paginación manual página por página...")
    return "manual"


def obtener_html_completo(driver, url: str) -> str:
    """Carga la página y retorna el HTML con TODOS los registros visibles."""
    driver.get(url)
    time.sleep(WAIT_SEGUNDOS)

    modo = mostrar_todas_las_filas(driver)

    if modo == "manual":
        return obtener_html_paginado(driver)

    return driver.page_source


def obtener_html_paginado(driver) -> str:
    """
    Recorre todas las páginas haciendo clic en 'Siguiente'
    y acumula los <tr> de cada página en una tabla unificada.
    """
    todas_filas_html = []
    pagina = 1

    while True:
        soup  = BeautifulSoup(driver.page_source, "html.parser")
        tabla = soup.find("table")
        if tabla:
            filas = tabla.find_all("tr")[1:]  # sin encabezado
            todas_filas_html.extend([str(f) for f in filas])
            print(f"    Página {pagina}: {len(filas)} filas | total acumulado: {len(todas_filas_html)}")

        try:
            siguiente = driver.find_element(
                By.CSS_SELECTOR,
                "a.paginate_button.next:not(.disabled), button.paginate_button.next:not(.disabled)"
            )
            siguiente.click()
            pagina += 1
            time.sleep(2)
        except Exception:
            print(f"    Fin de paginación en página {pagina}.")
            break

    # Reconstruir HTML con todas las filas
    soup_base  = BeautifulSoup(driver.page_source, "html.parser")
    tabla_base = soup_base.find("table")
    if tabla_base:
        tbody = tabla_base.find("tbody")
        if tbody:
            tbody.clear()
            for fila_html in todas_filas_html:
                tbody.append(BeautifulSoup(fila_html, "html.parser"))
    return str(soup_base)


def decodificar_b64(valor: str) -> str:
    try:
        return base64.b64decode(valor).decode("utf-8").strip()
    except Exception:
        return valor.strip()


def extraer_carrera(soup: BeautifulSoup) -> str:
    items = soup.select("ol li, ul.breadcrumb li")
    if items:
        return items[-1].get_text(strip=True)
    h1 = soup.find("h1")
    return h1.get_text(strip=True) if h1 else "DESCONOCIDA"


def normalizar_obs(val: str) -> str:
    v = val.strip().upper()
    if "ALCANZ" in v and "VACANTE" in v:
        return "ALCANZÓ VACANTE"
    if "AUSENTE" in v:
        return "AUSENTE"
    if "ART" in v:
        return "INHABILITADO (Art. 5)"
    return v if v else "SIN OBSERVACIÓN"


def parse_tabla(html: str, url: str) -> pd.DataFrame:
    soup  = BeautifulSoup(html, "html.parser")
    tabla = soup.find("table")

    if tabla is None:
        print(f"[AVISO] No se encontró tabla en: {url}")
        return pd.DataFrame()

    filas = []
    for tr in tabla.find_all("tr")[1:]:
        celdas = tr.find_all("td")
        if not celdas:
            continue

        codigo  = celdas[0].get_text(strip=True) if len(celdas) > 0 else ""

        nombre = ""
        if len(celdas) > 1:
            span = celdas[1].find("span", class_="obfuscated")
            nombre = decodificar_b64(span["data-auth"]) if (span and span.get("data-auth")) else celdas[1].get_text(strip=True)

        escuela = ""
        if len(celdas) > 2:
            span = celdas[2].find("span", class_="obfuscated")
            escuela = decodificar_b64(span["data-auth"]) if (span and span.get("data-auth")) else celdas[2].get_text(strip=True)

        puntaje = celdas[3].get("data-score", "").strip() if len(celdas) > 3 else ""
        merito  = celdas[4].get("data-merit", "").strip() if len(celdas) > 4 else ""
        obs     = celdas[5].get_text(strip=True)          if len(celdas) > 5 else ""

        filas.append({
            "codigo":            codigo,
            "apellidos_nombres": nombre,
            "escuela":           escuela,
            "puntaje":           puntaje,
            "merito_ep":         merito,
            "observacion":       obs,
        })

    if not filas:
        print(f"[AVISO] Sin filas en: {url}")
        return pd.DataFrame()

    df = pd.DataFrame(filas)
    df["codigo"] = df["codigo"].str.strip()
    df["apellidos_nombres"] = (
        df["apellidos_nombres"].str.strip().str.title()
        .str.replace(r"\s{2,}", " ", regex=True)
    )
    df["puntaje"]    = pd.to_numeric(df["puntaje"],  errors="coerce")
    df["merito_ep"]  = pd.to_numeric(df["merito_ep"], errors="coerce")
    df["observacion"] = df["observacion"].apply(normalizar_obs)
    df["carrera"]    = extraer_carrera(soup)
    df["url_fuente"] = url

    df = df.dropna(how="all").drop_duplicates(subset=["codigo"]).reset_index(drop=True)
    return df


def scrape_url(driver, url: str) -> pd.DataFrame:
    print(f"  Abriendo: {url}")
    html = obtener_html_completo(driver, url)
    df   = parse_tabla(html, url)
    print(f"  → {len(df)} registros extraídos.")
    return df


def guardar_csv(df: pd.DataFrame, ruta: Path):
    df.to_csv(ruta, index=False, encoding="utf-8-sig")
    print(f"\n✅ CSV guardado: {ruta.resolve()}")
    print(f"   {len(df)} filas  ×  {len(df.columns)} columnas")
    print("\n── Observaciones ──")
    print(df["observacion"].value_counts().to_string())
    if df["puntaje"].notna().any():
        print("\n── Estadísticas de puntaje ──")
        print(df["puntaje"].describe().round(3).to_string())


# ══════════════════════════════════════════════
# ▶ EJECUCIÓN
# ══════════════════════════════════════════════

driver = crear_driver()

try:
    if BATCH_RUTAS:
        dfs = []
        for ruta in BATCH_RUTAS:
            url = f"{BASE_URL}/{ruta.strip('/')}/results.html"
            df  = scrape_url(driver, url)
            if not df.empty:
                dfs.append(df)
            time.sleep(1.5)
        resultado = pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()
    else:
        resultado = scrape_url(driver, URL)
finally:
    driver.quit()
    print("[INFO] Navegador cerrado.")

if not resultado.empty:
    guardar_csv(resultado, Path(ARCHIVO_SALIDA))
    display(resultado.head(15))
else:
    print("❌ No se obtuvieron datos.")

  Abriendo: https://admision.unmsm.edu.pe/Website20262/A/153/results.html
  [✓] Paginación: forzado via DataTables JS API.
  → 57 registros extraídos.
[INFO] Navegador cerrado.

✅ CSV guardado: C:\Users\confe\OneDrive\Documentos\ANÁLISIS DE DATOS UNMSM 2026\resultados_unmsm_antropología.csv
   57 filas  ×  8 columnas

── Observaciones ──
observacion
ALCANZÓ VACANTE    38
SIN OBSERVACIÓN    18
AUSENTE             1

── Estadísticas de puntaje ──
count      56.000
mean      820.835
std       162.534
min       376.000
25%       704.562
50%       829.438
75%       925.750
max      1133.250


,codigo,apellidos_nombres,escuela,puntaje,merito_ep,observacion,carrera,url_fuente
0,708040,"Ambrosio Reyes, Yeraldine Paulet",ANTROPOLOGÍA,922.875,16.0,ALCANZÓ VACANTE,ANTROPOLOGÍA,https://admision.unmsm.edu.pe/Website20262/A/1...
1,707639,"Aquino Sanchez, Naomi Natali",ANTROPOLOGÍA,793.000,34.0,ALCANZÓ VACANTE,ANTROPOLOGÍA,https://admision.unmsm.edu.pe/Website20262/A/1...
2,721781,"Arce Quispe, Nicolas Aaron",ANTROPOLOGÍA,670.375,NaN,SIN OBSERVACIÓN,ANTROPOLOGÍA,https://admision.unmsm.edu.pe/Website20262/A/1...
3,717058,"Atahualpa Torres, Camila Andrea",ANTROPOLOGÍA,987.125,11.0,ALCANZÓ VACANTE,ANTROPOLOGÍA,https://admision.unmsm.edu.pe/Website20262/A/1...
4,829259,"Bances Barco, Victoria Mercedes",ANTROPOLOGÍA,376.000,NaN,SIN OBSERVACIÓN,ANTROPOLOGÍA,https://admision.unmsm.edu.pe/Website20262/A/1...
5,810085,"Bedon Echaccaya, Danna Alexandra",ANTROPOLOGÍA,807.250,31.0,ALCANZÓ VACANTE,ANTROPOLOGÍA,https://admision.unmsm.edu.pe/Website20262/A/1...
6,840236,"Borja Principe, Sofia Victoria",ANTROPOLOGÍA,885.000,21.0,ALCANZÓ VACANTE,ANTROPOLOGÍA,https://admision.unmsm.edu.pe/Website20262/A/1...
7,771489,"Brito Robles, Angela Lupita Analhy",ANTROPOLOGÍA,897.500,20.0,ALCANZÓ VACANTE,ANTROPOLOGÍA,https://admision.unmsm.edu.pe/Website20262/A/1...
8,749295,"Carranza Quillca, Abrahan Eduardo",ANTROPOLOGÍA,834.625,28.0,ALCANZÓ VACANTE,ANTROPOLOGÍA,https://admision.unmsm.edu.pe/Website20262/A/1...
9,753059,"Castro Condori, Sharill Lucero Celeste",ANTROPOLOGÍA,717.625,NaN,SIN OBSERVACIÓN,ANTROPOLOGÍA,https://admision.unmsm.edu.pe/Website20262/A/1...


In [32]:
import pandas as pd

df = pd.read_csv("resultados_unmsm_antropología.csv")

vacantes = df[df["observacion"] == "ALCANZÓ VACANTE"]

max_p = vacantes["puntaje"].max()
min_p = vacantes["puntaje"].min()

print(f"Carrera: {df['carrera'].iloc[0]}")
print(f"Vacantes obtenidas: {len(vacantes)}")
print(f"Puntaje máximo: {max_p}")
print(f"Puntaje mínimo: {min_p}")

display(vacantes[["codigo","apellidos_nombres","puntaje","merito_ep"]]
        .sort_values("puntaje", ascending=False)
        .reset_index(drop=True))

Carrera: ANTROPOLOGÍA
Vacantes obtenidas: 38
Puntaje máximo: 1133.25
Puntaje mínimo: 763.25


,codigo,apellidos_nombres,puntaje,merito_ep
0,800566,"Lara Paniura, Cristian José",1133.250,1.0
1,757113,"Huamani Vicente, Estela Cristin",1132.250,2.0
2,725403,"Llamocca Espinoza, Adriano Leonardo",1052.750,3.0
3,768216,"Meza Nuñez, Ricardo Rodrigo",1048.125,4.0
4,753397,"Ccoyllo Rivera, Fritzia Fattima",1038.500,5.0
5,808535,"Gonzales Ochoa, Ana Del Sol Micaela",1027.125,6.0
6,828392,"Rodriguez Espinoza, Silvio Facundo",1017.875,7.0
7,706291,"Puma Gomez, Kiara Yamilé",1012.250,8.0
8,736926,"Quintana Chero, Celeste Alely",1011.375,9.0
9,690991,"Ricra Huamani, Angely Selena",1000.000,10.0


In [33]:
import time
import base64
import pandas as pd
from bs4 import BeautifulSoup
from pathlib import Path

from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait, Select
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager

# ══════════════════════════════════════════════
# ▶ CONFIGURACIÓN
# ══════════════════════════════════════════════

URL            = "https://admision.unmsm.edu.pe/Website20262/A/154/results.html"
BATCH_RUTAS    = []   # ejemplo: ["A/091", "A/092"]
ARCHIVO_SALIDA = "resultados_unmsm_ARQUEOLOGÍA.csv"
BASE_URL       = "https://admision.unmsm.edu.pe/Website20262"
WAIT_SEGUNDOS  = 8

# ══════════════════════════════════════════════


def crear_driver():
    opciones = Options()
    opciones.add_argument("--headless=new")
    opciones.add_argument("--no-sandbox")
    opciones.add_argument("--disable-dev-shm-usage")
    opciones.add_argument("--disable-gpu")
    opciones.add_argument("--window-size=1920,1080")
    opciones.add_argument(
        "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36"
    )
    return webdriver.Chrome(
        service=Service(ChromeDriverManager().install()),
        options=opciones
    )


def mostrar_todas_las_filas(driver):
    """
    Intenta mostrar TODOS los registros de una vez usando 3 estrategias:
    1. Cambiar el select de longitud de DataTables a -1 (All) vía Selenium
    2. Forzarlo por JavaScript directamente sobre la instancia DataTable
    3. Si no hay selector, navegar página por página
    """
    wait = WebDriverWait(driver, 10)

    # ── Estrategia 1: select de longitud (típico en DataTables) ──────────
    try:
        select_el = wait.until(
            EC.presence_of_element_located((By.CSS_SELECTOR, "select[name*='length'], select[name*='DataTables']"))
        )
        select = Select(select_el)
        try:
            select.select_by_value("-1")
            print("  [✓] Paginación: seleccionado 'Todos' en el select.")
        except Exception:
            opciones_vals = [o.get_attribute("value") for o in select.options]
            select.select_by_value(opciones_vals[-1])
            print(f"  [✓] Paginación: seleccionado valor máximo ({opciones_vals[-1]}) en el select.")
        time.sleep(3)
        return "select"
    except Exception:
        pass

    # ── Estrategia 2: forzar via JavaScript DataTables API ───────────────
    try:
        driver.execute_script("""
            var tables = $.fn.dataTable ? $.fn.dataTable.tables() : [];
            if (tables.length > 0) {
                $(tables[0]).DataTable().page.len(-1).draw();
            }
        """)
        time.sleep(3)
        print("  [✓] Paginación: forzado via DataTables JS API.")
        return "js"
    except Exception:
        pass

    # ── Estrategia 3: paginación manual (clic en "Siguiente") ────────────
    print("  [!] Usando paginación manual página por página...")
    return "manual"


def obtener_html_completo(driver, url: str) -> str:
    """Carga la página y retorna el HTML con TODOS los registros visibles."""
    driver.get(url)
    time.sleep(WAIT_SEGUNDOS)

    modo = mostrar_todas_las_filas(driver)

    if modo == "manual":
        return obtener_html_paginado(driver)

    return driver.page_source


def obtener_html_paginado(driver) -> str:
    """
    Recorre todas las páginas haciendo clic en 'Siguiente'
    y acumula los <tr> de cada página en una tabla unificada.
    """
    todas_filas_html = []
    pagina = 1

    while True:
        soup  = BeautifulSoup(driver.page_source, "html.parser")
        tabla = soup.find("table")
        if tabla:
            filas = tabla.find_all("tr")[1:]  # sin encabezado
            todas_filas_html.extend([str(f) for f in filas])
            print(f"    Página {pagina}: {len(filas)} filas | total acumulado: {len(todas_filas_html)}")

        try:
            siguiente = driver.find_element(
                By.CSS_SELECTOR,
                "a.paginate_button.next:not(.disabled), button.paginate_button.next:not(.disabled)"
            )
            siguiente.click()
            pagina += 1
            time.sleep(2)
        except Exception:
            print(f"    Fin de paginación en página {pagina}.")
            break

    # Reconstruir HTML con todas las filas
    soup_base  = BeautifulSoup(driver.page_source, "html.parser")
    tabla_base = soup_base.find("table")
    if tabla_base:
        tbody = tabla_base.find("tbody")
        if tbody:
            tbody.clear()
            for fila_html in todas_filas_html:
                tbody.append(BeautifulSoup(fila_html, "html.parser"))
    return str(soup_base)


def decodificar_b64(valor: str) -> str:
    try:
        return base64.b64decode(valor).decode("utf-8").strip()
    except Exception:
        return valor.strip()


def extraer_carrera(soup: BeautifulSoup) -> str:
    items = soup.select("ol li, ul.breadcrumb li")
    if items:
        return items[-1].get_text(strip=True)
    h1 = soup.find("h1")
    return h1.get_text(strip=True) if h1 else "DESCONOCIDA"


def normalizar_obs(val: str) -> str:
    v = val.strip().upper()
    if "ALCANZ" in v and "VACANTE" in v:
        return "ALCANZÓ VACANTE"
    if "AUSENTE" in v:
        return "AUSENTE"
    if "ART" in v:
        return "INHABILITADO (Art. 5)"
    return v if v else "SIN OBSERVACIÓN"


def parse_tabla(html: str, url: str) -> pd.DataFrame:
    soup  = BeautifulSoup(html, "html.parser")
    tabla = soup.find("table")

    if tabla is None:
        print(f"[AVISO] No se encontró tabla en: {url}")
        return pd.DataFrame()

    filas = []
    for tr in tabla.find_all("tr")[1:]:
        celdas = tr.find_all("td")
        if not celdas:
            continue

        codigo  = celdas[0].get_text(strip=True) if len(celdas) > 0 else ""

        nombre = ""
        if len(celdas) > 1:
            span = celdas[1].find("span", class_="obfuscated")
            nombre = decodificar_b64(span["data-auth"]) if (span and span.get("data-auth")) else celdas[1].get_text(strip=True)

        escuela = ""
        if len(celdas) > 2:
            span = celdas[2].find("span", class_="obfuscated")
            escuela = decodificar_b64(span["data-auth"]) if (span and span.get("data-auth")) else celdas[2].get_text(strip=True)

        puntaje = celdas[3].get("data-score", "").strip() if len(celdas) > 3 else ""
        merito  = celdas[4].get("data-merit", "").strip() if len(celdas) > 4 else ""
        obs     = celdas[5].get_text(strip=True)          if len(celdas) > 5 else ""

        filas.append({
            "codigo":            codigo,
            "apellidos_nombres": nombre,
            "escuela":           escuela,
            "puntaje":           puntaje,
            "merito_ep":         merito,
            "observacion":       obs,
        })

    if not filas:
        print(f"[AVISO] Sin filas en: {url}")
        return pd.DataFrame()

    df = pd.DataFrame(filas)
    df["codigo"] = df["codigo"].str.strip()
    df["apellidos_nombres"] = (
        df["apellidos_nombres"].str.strip().str.title()
        .str.replace(r"\s{2,}", " ", regex=True)
    )
    df["puntaje"]    = pd.to_numeric(df["puntaje"],  errors="coerce")
    df["merito_ep"]  = pd.to_numeric(df["merito_ep"], errors="coerce")
    df["observacion"] = df["observacion"].apply(normalizar_obs)
    df["carrera"]    = extraer_carrera(soup)
    df["url_fuente"] = url

    df = df.dropna(how="all").drop_duplicates(subset=["codigo"]).reset_index(drop=True)
    return df


def scrape_url(driver, url: str) -> pd.DataFrame:
    print(f"  Abriendo: {url}")
    html = obtener_html_completo(driver, url)
    df   = parse_tabla(html, url)
    print(f"  → {len(df)} registros extraídos.")
    return df


def guardar_csv(df: pd.DataFrame, ruta: Path):
    df.to_csv(ruta, index=False, encoding="utf-8-sig")
    print(f"\n✅ CSV guardado: {ruta.resolve()}")
    print(f"   {len(df)} filas  ×  {len(df.columns)} columnas")
    print("\n── Observaciones ──")
    print(df["observacion"].value_counts().to_string())
    if df["puntaje"].notna().any():
        print("\n── Estadísticas de puntaje ──")
        print(df["puntaje"].describe().round(3).to_string())


# ══════════════════════════════════════════════
# ▶ EJECUCIÓN
# ══════════════════════════════════════════════

driver = crear_driver()

try:
    if BATCH_RUTAS:
        dfs = []
        for ruta in BATCH_RUTAS:
            url = f"{BASE_URL}/{ruta.strip('/')}/results.html"
            df  = scrape_url(driver, url)
            if not df.empty:
                dfs.append(df)
            time.sleep(1.5)
        resultado = pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()
    else:
        resultado = scrape_url(driver, URL)
finally:
    driver.quit()
    print("[INFO] Navegador cerrado.")

if not resultado.empty:
    guardar_csv(resultado, Path(ARCHIVO_SALIDA))
    display(resultado.head(15))
else:
    print("❌ No se obtuvieron datos.")

  Abriendo: https://admision.unmsm.edu.pe/Website20262/A/154/results.html
  [✓] Paginación: forzado via DataTables JS API.
  → 73 registros extraídos.
[INFO] Navegador cerrado.

✅ CSV guardado: C:\Users\confe\OneDrive\Documentos\ANÁLISIS DE DATOS UNMSM 2026\resultados_unmsm_ARQUEOLOGÍA.csv
   73 filas  ×  8 columnas

── Observaciones ──
observacion
ALCANZÓ VACANTE          40
SIN OBSERVACIÓN          32
INHABILITADO (Art. 5)     1

── Estadísticas de puntaje ──
count      73.000
mean      850.885
std       168.473
min       464.250
25%       772.625
50%       852.375
75%       954.000
max      1227.500


,codigo,apellidos_nombres,escuela,puntaje,merito_ep,observacion,carrera,url_fuente
0,754619,"Alcalde Rosales, Alheli Sharon",ARQUEOLOGÍA,811.625,NaN,SIN OBSERVACIÓN,ARQUEOLOGÍA,https://admision.unmsm.edu.pe/Website20262/A/1...
1,690701,"Aliaga Luyo, Adrian Valentino",ARQUEOLOGÍA,772.625,NaN,SIN OBSERVACIÓN,ARQUEOLOGÍA,https://admision.unmsm.edu.pe/Website20262/A/1...
2,715852,"Alvarez Insil, Jhon Agustin",ARQUEOLOGÍA,1047.000,10.0,ALCANZÓ VACANTE,ARQUEOLOGÍA,https://admision.unmsm.edu.pe/Website20262/A/1...
3,729448,"Antay Caporal, Paola Virginia",ARQUEOLOGÍA,826.125,40.0,ALCANZÓ VACANTE,ARQUEOLOGÍA,https://admision.unmsm.edu.pe/Website20262/A/1...
4,817977,"Arana Guillen, Jhon Fredy",ARQUEOLOGÍA,927.750,19.0,ALCANZÓ VACANTE,ARQUEOLOGÍA,https://admision.unmsm.edu.pe/Website20262/A/1...
5,701457,"Arca Fiestas, Josué Ricardo",ARQUEOLOGÍA,814.125,NaN,SIN OBSERVACIÓN,ARQUEOLOGÍA,https://admision.unmsm.edu.pe/Website20262/A/1...
6,758153,"Ataupillco Mejia, Mathias Francisco",ARQUEOLOGÍA,775.875,NaN,SIN OBSERVACIÓN,ARQUEOLOGÍA,https://admision.unmsm.edu.pe/Website20262/A/1...
7,691566,"Barahona Mallaupoma, Ariana Araceli",ARQUEOLOGÍA,1067.000,8.0,ALCANZÓ VACANTE,ARQUEOLOGÍA,https://admision.unmsm.edu.pe/Website20262/A/1...
8,780305,"Barbarán Medina, Nicolle Valentina",ARQUEOLOGÍA,856.875,33.0,ALCANZÓ VACANTE,ARQUEOLOGÍA,https://admision.unmsm.edu.pe/Website20262/A/1...
9,722977,"Bedon Chauca, Frank Danilo",ARQUEOLOGÍA,896.125,24.0,ALCANZÓ VACANTE,ARQUEOLOGÍA,https://admision.unmsm.edu.pe/Website20262/A/1...


In [34]:
import pandas as pd

df = pd.read_csv("resultados_unmsm_ARQUEOLOGÍA.csv")

vacantes = df[df["observacion"] == "ALCANZÓ VACANTE"]

max_p = vacantes["puntaje"].max()
min_p = vacantes["puntaje"].min()

print(f"Carrera: {df['carrera'].iloc[0]}")
print(f"Vacantes obtenidas: {len(vacantes)}")
print(f"Puntaje máximo: {max_p}")
print(f"Puntaje mínimo: {min_p}")

display(vacantes[["codigo","apellidos_nombres","puntaje","merito_ep"]]
        .sort_values("puntaje", ascending=False)
        .reset_index(drop=True))

Carrera: ARQUEOLOGÍA
Vacantes obtenidas: 40
Puntaje máximo: 1227.5
Puntaje mínimo: 826.125


,codigo,apellidos_nombres,puntaje,merito_ep
0,700710,"Siancas Quispe, Lucas Nahum",1227.500,1.0
1,739126,"Huaman Rivera, Alfredo Joaquin",1176.000,2.0
2,732919,"Mamani Huaman, Maria Fernanda",1168.500,3.0
3,732167,"Chavez Saldaña, Amira Ayyubs",1149.625,4.0
4,811692,"Neyra Medina, Wilian Alexander",1140.000,5.0
5,713235,"Castellanos Salvatierra, Joseph Anthony",1137.250,6.0
6,759124,"Zúñiga Mendizábal, Jose Mario",1112.750,7.0
7,691566,"Barahona Mallaupoma, Ariana Araceli",1067.000,8.0
8,729742,"Valencia Santillana, Alexis Jarod",1066.500,9.0
9,715852,"Alvarez Insil, Jhon Agustin",1047.000,10.0


In [35]:
import time
import base64
import pandas as pd
from bs4 import BeautifulSoup
from pathlib import Path

from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait, Select
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager

# ══════════════════════════════════════════════
# ▶ CONFIGURACIÓN
# ══════════════════════════════════════════════

URL            = "https://admision.unmsm.edu.pe/Website20262/A/169/results.html"
BATCH_RUTAS    = []   # ejemplo: ["A/091", "A/092"]
ARCHIVO_SALIDA = "resultados_unmsm_ARQUITECTURA.csv"
BASE_URL       = "https://admision.unmsm.edu.pe/Website20262"
WAIT_SEGUNDOS  = 8

# ══════════════════════════════════════════════


def crear_driver():
    opciones = Options()
    opciones.add_argument("--headless=new")
    opciones.add_argument("--no-sandbox")
    opciones.add_argument("--disable-dev-shm-usage")
    opciones.add_argument("--disable-gpu")
    opciones.add_argument("--window-size=1920,1080")
    opciones.add_argument(
        "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36"
    )
    return webdriver.Chrome(
        service=Service(ChromeDriverManager().install()),
        options=opciones
    )


def mostrar_todas_las_filas(driver):
    """
    Intenta mostrar TODOS los registros de una vez usando 3 estrategias:
    1. Cambiar el select de longitud de DataTables a -1 (All) vía Selenium
    2. Forzarlo por JavaScript directamente sobre la instancia DataTable
    3. Si no hay selector, navegar página por página
    """
    wait = WebDriverWait(driver, 10)

    # ── Estrategia 1: select de longitud (típico en DataTables) ──────────
    try:
        select_el = wait.until(
            EC.presence_of_element_located((By.CSS_SELECTOR, "select[name*='length'], select[name*='DataTables']"))
        )
        select = Select(select_el)
        try:
            select.select_by_value("-1")
            print("  [✓] Paginación: seleccionado 'Todos' en el select.")
        except Exception:
            opciones_vals = [o.get_attribute("value") for o in select.options]
            select.select_by_value(opciones_vals[-1])
            print(f"  [✓] Paginación: seleccionado valor máximo ({opciones_vals[-1]}) en el select.")
        time.sleep(3)
        return "select"
    except Exception:
        pass

    # ── Estrategia 2: forzar via JavaScript DataTables API ───────────────
    try:
        driver.execute_script("""
            var tables = $.fn.dataTable ? $.fn.dataTable.tables() : [];
            if (tables.length > 0) {
                $(tables[0]).DataTable().page.len(-1).draw();
            }
        """)
        time.sleep(3)
        print("  [✓] Paginación: forzado via DataTables JS API.")
        return "js"
    except Exception:
        pass

    # ── Estrategia 3: paginación manual (clic en "Siguiente") ────────────
    print("  [!] Usando paginación manual página por página...")
    return "manual"


def obtener_html_completo(driver, url: str) -> str:
    """Carga la página y retorna el HTML con TODOS los registros visibles."""
    driver.get(url)
    time.sleep(WAIT_SEGUNDOS)

    modo = mostrar_todas_las_filas(driver)

    if modo == "manual":
        return obtener_html_paginado(driver)

    return driver.page_source


def obtener_html_paginado(driver) -> str:
    """
    Recorre todas las páginas haciendo clic en 'Siguiente'
    y acumula los <tr> de cada página en una tabla unificada.
    """
    todas_filas_html = []
    pagina = 1

    while True:
        soup  = BeautifulSoup(driver.page_source, "html.parser")
        tabla = soup.find("table")
        if tabla:
            filas = tabla.find_all("tr")[1:]  # sin encabezado
            todas_filas_html.extend([str(f) for f in filas])
            print(f"    Página {pagina}: {len(filas)} filas | total acumulado: {len(todas_filas_html)}")

        try:
            siguiente = driver.find_element(
                By.CSS_SELECTOR,
                "a.paginate_button.next:not(.disabled), button.paginate_button.next:not(.disabled)"
            )
            siguiente.click()
            pagina += 1
            time.sleep(2)
        except Exception:
            print(f"    Fin de paginación en página {pagina}.")
            break

    # Reconstruir HTML con todas las filas
    soup_base  = BeautifulSoup(driver.page_source, "html.parser")
    tabla_base = soup_base.find("table")
    if tabla_base:
        tbody = tabla_base.find("tbody")
        if tbody:
            tbody.clear()
            for fila_html in todas_filas_html:
                tbody.append(BeautifulSoup(fila_html, "html.parser"))
    return str(soup_base)


def decodificar_b64(valor: str) -> str:
    try:
        return base64.b64decode(valor).decode("utf-8").strip()
    except Exception:
        return valor.strip()


def extraer_carrera(soup: BeautifulSoup) -> str:
    items = soup.select("ol li, ul.breadcrumb li")
    if items:
        return items[-1].get_text(strip=True)
    h1 = soup.find("h1")
    return h1.get_text(strip=True) if h1 else "DESCONOCIDA"


def normalizar_obs(val: str) -> str:
    v = val.strip().upper()
    if "ALCANZ" in v and "VACANTE" in v:
        return "ALCANZÓ VACANTE"
    if "AUSENTE" in v:
        return "AUSENTE"
    if "ART" in v:
        return "INHABILITADO (Art. 5)"
    return v if v else "SIN OBSERVACIÓN"


def parse_tabla(html: str, url: str) -> pd.DataFrame:
    soup  = BeautifulSoup(html, "html.parser")
    tabla = soup.find("table")

    if tabla is None:
        print(f"[AVISO] No se encontró tabla en: {url}")
        return pd.DataFrame()

    filas = []
    for tr in tabla.find_all("tr")[1:]:
        celdas = tr.find_all("td")
        if not celdas:
            continue

        codigo  = celdas[0].get_text(strip=True) if len(celdas) > 0 else ""

        nombre = ""
        if len(celdas) > 1:
            span = celdas[1].find("span", class_="obfuscated")
            nombre = decodificar_b64(span["data-auth"]) if (span and span.get("data-auth")) else celdas[1].get_text(strip=True)

        escuela = ""
        if len(celdas) > 2:
            span = celdas[2].find("span", class_="obfuscated")
            escuela = decodificar_b64(span["data-auth"]) if (span and span.get("data-auth")) else celdas[2].get_text(strip=True)

        puntaje = celdas[3].get("data-score", "").strip() if len(celdas) > 3 else ""
        merito  = celdas[4].get("data-merit", "").strip() if len(celdas) > 4 else ""
        obs     = celdas[5].get_text(strip=True)          if len(celdas) > 5 else ""

        filas.append({
            "codigo":            codigo,
            "apellidos_nombres": nombre,
            "escuela":           escuela,
            "puntaje":           puntaje,
            "merito_ep":         merito,
            "observacion":       obs,
        })

    if not filas:
        print(f"[AVISO] Sin filas en: {url}")
        return pd.DataFrame()

    df = pd.DataFrame(filas)
    df["codigo"] = df["codigo"].str.strip()
    df["apellidos_nombres"] = (
        df["apellidos_nombres"].str.strip().str.title()
        .str.replace(r"\s{2,}", " ", regex=True)
    )
    df["puntaje"]    = pd.to_numeric(df["puntaje"],  errors="coerce")
    df["merito_ep"]  = pd.to_numeric(df["merito_ep"], errors="coerce")
    df["observacion"] = df["observacion"].apply(normalizar_obs)
    df["carrera"]    = extraer_carrera(soup)
    df["url_fuente"] = url

    df = df.dropna(how="all").drop_duplicates(subset=["codigo"]).reset_index(drop=True)
    return df


def scrape_url(driver, url: str) -> pd.DataFrame:
    print(f"  Abriendo: {url}")
    html = obtener_html_completo(driver, url)
    df   = parse_tabla(html, url)
    print(f"  → {len(df)} registros extraídos.")
    return df


def guardar_csv(df: pd.DataFrame, ruta: Path):
    df.to_csv(ruta, index=False, encoding="utf-8-sig")
    print(f"\n✅ CSV guardado: {ruta.resolve()}")
    print(f"   {len(df)} filas  ×  {len(df.columns)} columnas")
    print("\n── Observaciones ──")
    print(df["observacion"].value_counts().to_string())
    if df["puntaje"].notna().any():
        print("\n── Estadísticas de puntaje ──")
        print(df["puntaje"].describe().round(3).to_string())


# ══════════════════════════════════════════════
# ▶ EJECUCIÓN
# ══════════════════════════════════════════════

driver = crear_driver()

try:
    if BATCH_RUTAS:
        dfs = []
        for ruta in BATCH_RUTAS:
            url = f"{BASE_URL}/{ruta.strip('/')}/results.html"
            df  = scrape_url(driver, url)
            if not df.empty:
                dfs.append(df)
            time.sleep(1.5)
        resultado = pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()
    else:
        resultado = scrape_url(driver, URL)
finally:
    driver.quit()
    print("[INFO] Navegador cerrado.")

if not resultado.empty:
    guardar_csv(resultado, Path(ARCHIVO_SALIDA))
    display(resultado.head(15))
else:
    print("❌ No se obtuvieron datos.")

  Abriendo: https://admision.unmsm.edu.pe/Website20262/A/169/results.html
  [✓] Paginación: forzado via DataTables JS API.
  → 514 registros extraídos.
[INFO] Navegador cerrado.

✅ CSV guardado: C:\Users\confe\OneDrive\Documentos\ANÁLISIS DE DATOS UNMSM 2026\resultados_unmsm_ARQUITECTURA.csv
   514 filas  ×  8 columnas

── Observaciones ──
observacion
SIN OBSERVACIÓN          479
ALCANZÓ VACANTE           26
INHABILITADO (Art. 5)      6
AUSENTE                    3

── Estadísticas de puntaje ──
count     511.000
mean      863.529
std       198.377
min       347.125
25%       711.562
50%       857.375
75%       997.750
max      1565.500


,codigo,apellidos_nombres,escuela,puntaje,merito_ep,observacion,carrera,url_fuente
0,345985,"Acosta Cayetano, Jhony Daniel",ARQUITECTURA Y URBANISMO,1407.250,3.0,ALCANZÓ VACANTE,ARQUITECTURA Y URBANISMO,https://admision.unmsm.edu.pe/Website20262/A/1...
1,356469,"Agreda Ambrosio, José Franco",ARQUITECTURA Y URBANISMO,818.625,NaN,SIN OBSERVACIÓN,ARQUITECTURA Y URBANISMO,https://admision.unmsm.edu.pe/Website20262/A/1...
2,321037,"Aguiar Mora, Karolay Damiana",ARQUITECTURA Y URBANISMO,737.000,NaN,SIN OBSERVACIÓN,ARQUITECTURA Y URBANISMO,https://admision.unmsm.edu.pe/Website20262/A/1...
3,356951,"Aguilar Galdos, Alessandra Lucciana",ARQUITECTURA Y URBANISMO,692.500,NaN,SIN OBSERVACIÓN,ARQUITECTURA Y URBANISMO,https://admision.unmsm.edu.pe/Website20262/A/1...
4,336566,"Aguilar Ore, Vanessa Emily",ARQUITECTURA Y URBANISMO,768.625,NaN,SIN OBSERVACIÓN,ARQUITECTURA Y URBANISMO,https://admision.unmsm.edu.pe/Website20262/A/1...
5,395074,"Aguilar Rojas, Reynaldo Duvan",ARQUITECTURA Y URBANISMO,1175.500,NaN,SIN OBSERVACIÓN,ARQUITECTURA Y URBANISMO,https://admision.unmsm.edu.pe/Website20262/A/1...
6,348739,"Aguirre Illa, Lucy",ARQUITECTURA Y URBANISMO,958.000,NaN,SIN OBSERVACIÓN,ARQUITECTURA Y URBANISMO,https://admision.unmsm.edu.pe/Website20262/A/1...
7,369073,"Alarco Macedo, Luz Maria",ARQUITECTURA Y URBANISMO,821.625,NaN,SIN OBSERVACIÓN,ARQUITECTURA Y URBANISMO,https://admision.unmsm.edu.pe/Website20262/A/1...
8,322106,"Alberca Lima, Maryore Yosani",ARQUITECTURA Y URBANISMO,587.625,NaN,SIN OBSERVACIÓN,ARQUITECTURA Y URBANISMO,https://admision.unmsm.edu.pe/Website20262/A/1...
9,321916,"Alcantara Juipa, Angela Dayana",ARQUITECTURA Y URBANISMO,801.000,NaN,SIN OBSERVACIÓN,ARQUITECTURA Y URBANISMO,https://admision.unmsm.edu.pe/Website20262/A/1...


In [36]:
import pandas as pd

df = pd.read_csv("resultados_unmsm_ARQUITECTURA.csv")

vacantes = df[df["observacion"] == "ALCANZÓ VACANTE"]

max_p = vacantes["puntaje"].max()
min_p = vacantes["puntaje"].min()

print(f"Carrera: {df['carrera'].iloc[0]}")
print(f"Vacantes obtenidas: {len(vacantes)}")
print(f"Puntaje máximo: {max_p}")
print(f"Puntaje mínimo: {min_p}")

display(vacantes[["codigo","apellidos_nombres","puntaje","merito_ep"]]
        .sort_values("puntaje", ascending=False)
        .reset_index(drop=True))

Carrera: ARQUITECTURA Y URBANISMO
Vacantes obtenidas: 26
Puntaje máximo: 1565.5
Puntaje mínimo: 1187.5


,codigo,apellidos_nombres,puntaje,merito_ep
0,348258,"Lazo Tovar, Maximo Valentino",1565.500,1.0
1,357116,"Villanueva Gonzalo, Mia Aleli",1506.125,2.0
2,345985,"Acosta Cayetano, Jhony Daniel",1407.250,3.0
3,360801,"Mayta Espinoza, Diego Alonso",1342.250,4.0
4,386863,"Pfuro Palomino, Johany Belen",1339.375,5.0
5,347424,"Porras Huarancca, Paloma Steysi",1296.000,6.0
6,394207,"Medrano Torres, Luis Daniel",1279.375,7.0
7,372398,"Fernández Del Castillo, Lindsay Mariana",1275.875,8.0
8,329632,"Roque Maldonado, Valeria Alejandra",1269.375,9.0
9,359456,"Huatuco Flores, Angie Belen",1266.875,10.0


In [37]:
import time
import base64
import pandas as pd
from bs4 import BeautifulSoup
from pathlib import Path

from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait, Select
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager

# ══════════════════════════════════════════════
# ▶ CONFIGURACIÓN
# ══════════════════════════════════════════════

URL            = "https://admision.unmsm.edu.pe/Website20262/A/036/results.html"
BATCH_RUTAS    = []   # ejemplo: ["A/091", "A/092"]
ARCHIVO_SALIDA = "resultados_unmsm_ARTE.csv"
BASE_URL       = "https://admision.unmsm.edu.pe/Website20262"
WAIT_SEGUNDOS  = 8

# ══════════════════════════════════════════════


def crear_driver():
    opciones = Options()
    opciones.add_argument("--headless=new")
    opciones.add_argument("--no-sandbox")
    opciones.add_argument("--disable-dev-shm-usage")
    opciones.add_argument("--disable-gpu")
    opciones.add_argument("--window-size=1920,1080")
    opciones.add_argument(
        "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36"
    )
    return webdriver.Chrome(
        service=Service(ChromeDriverManager().install()),
        options=opciones
    )


def mostrar_todas_las_filas(driver):
    """
    Intenta mostrar TODOS los registros de una vez usando 3 estrategias:
    1. Cambiar el select de longitud de DataTables a -1 (All) vía Selenium
    2. Forzarlo por JavaScript directamente sobre la instancia DataTable
    3. Si no hay selector, navegar página por página
    """
    wait = WebDriverWait(driver, 10)

    # ── Estrategia 1: select de longitud (típico en DataTables) ──────────
    try:
        select_el = wait.until(
            EC.presence_of_element_located((By.CSS_SELECTOR, "select[name*='length'], select[name*='DataTables']"))
        )
        select = Select(select_el)
        try:
            select.select_by_value("-1")
            print("  [✓] Paginación: seleccionado 'Todos' en el select.")
        except Exception:
            opciones_vals = [o.get_attribute("value") for o in select.options]
            select.select_by_value(opciones_vals[-1])
            print(f"  [✓] Paginación: seleccionado valor máximo ({opciones_vals[-1]}) en el select.")
        time.sleep(3)
        return "select"
    except Exception:
        pass

    # ── Estrategia 2: forzar via JavaScript DataTables API ───────────────
    try:
        driver.execute_script("""
            var tables = $.fn.dataTable ? $.fn.dataTable.tables() : [];
            if (tables.length > 0) {
                $(tables[0]).DataTable().page.len(-1).draw();
            }
        """)
        time.sleep(3)
        print("  [✓] Paginación: forzado via DataTables JS API.")
        return "js"
    except Exception:
        pass

    # ── Estrategia 3: paginación manual (clic en "Siguiente") ────────────
    print("  [!] Usando paginación manual página por página...")
    return "manual"


def obtener_html_completo(driver, url: str) -> str:
    """Carga la página y retorna el HTML con TODOS los registros visibles."""
    driver.get(url)
    time.sleep(WAIT_SEGUNDOS)

    modo = mostrar_todas_las_filas(driver)

    if modo == "manual":
        return obtener_html_paginado(driver)

    return driver.page_source


def obtener_html_paginado(driver) -> str:
    """
    Recorre todas las páginas haciendo clic en 'Siguiente'
    y acumula los <tr> de cada página en una tabla unificada.
    """
    todas_filas_html = []
    pagina = 1

    while True:
        soup  = BeautifulSoup(driver.page_source, "html.parser")
        tabla = soup.find("table")
        if tabla:
            filas = tabla.find_all("tr")[1:]  # sin encabezado
            todas_filas_html.extend([str(f) for f in filas])
            print(f"    Página {pagina}: {len(filas)} filas | total acumulado: {len(todas_filas_html)}")

        try:
            siguiente = driver.find_element(
                By.CSS_SELECTOR,
                "a.paginate_button.next:not(.disabled), button.paginate_button.next:not(.disabled)"
            )
            siguiente.click()
            pagina += 1
            time.sleep(2)
        except Exception:
            print(f"    Fin de paginación en página {pagina}.")
            break

    # Reconstruir HTML con todas las filas
    soup_base  = BeautifulSoup(driver.page_source, "html.parser")
    tabla_base = soup_base.find("table")
    if tabla_base:
        tbody = tabla_base.find("tbody")
        if tbody:
            tbody.clear()
            for fila_html in todas_filas_html:
                tbody.append(BeautifulSoup(fila_html, "html.parser"))
    return str(soup_base)


def decodificar_b64(valor: str) -> str:
    try:
        return base64.b64decode(valor).decode("utf-8").strip()
    except Exception:
        return valor.strip()


def extraer_carrera(soup: BeautifulSoup) -> str:
    items = soup.select("ol li, ul.breadcrumb li")
    if items:
        return items[-1].get_text(strip=True)
    h1 = soup.find("h1")
    return h1.get_text(strip=True) if h1 else "DESCONOCIDA"


def normalizar_obs(val: str) -> str:
    v = val.strip().upper()
    if "ALCANZ" in v and "VACANTE" in v:
        return "ALCANZÓ VACANTE"
    if "AUSENTE" in v:
        return "AUSENTE"
    if "ART" in v:
        return "INHABILITADO (Art. 5)"
    return v if v else "SIN OBSERVACIÓN"


def parse_tabla(html: str, url: str) -> pd.DataFrame:
    soup  = BeautifulSoup(html, "html.parser")
    tabla = soup.find("table")

    if tabla is None:
        print(f"[AVISO] No se encontró tabla en: {url}")
        return pd.DataFrame()

    filas = []
    for tr in tabla.find_all("tr")[1:]:
        celdas = tr.find_all("td")
        if not celdas:
            continue

        codigo  = celdas[0].get_text(strip=True) if len(celdas) > 0 else ""

        nombre = ""
        if len(celdas) > 1:
            span = celdas[1].find("span", class_="obfuscated")
            nombre = decodificar_b64(span["data-auth"]) if (span and span.get("data-auth")) else celdas[1].get_text(strip=True)

        escuela = ""
        if len(celdas) > 2:
            span = celdas[2].find("span", class_="obfuscated")
            escuela = decodificar_b64(span["data-auth"]) if (span and span.get("data-auth")) else celdas[2].get_text(strip=True)

        puntaje = celdas[3].get("data-score", "").strip() if len(celdas) > 3 else ""
        merito  = celdas[4].get("data-merit", "").strip() if len(celdas) > 4 else ""
        obs     = celdas[5].get_text(strip=True)          if len(celdas) > 5 else ""

        filas.append({
            "codigo":            codigo,
            "apellidos_nombres": nombre,
            "escuela":           escuela,
            "puntaje":           puntaje,
            "merito_ep":         merito,
            "observacion":       obs,
        })

    if not filas:
        print(f"[AVISO] Sin filas en: {url}")
        return pd.DataFrame()

    df = pd.DataFrame(filas)
    df["codigo"] = df["codigo"].str.strip()
    df["apellidos_nombres"] = (
        df["apellidos_nombres"].str.strip().str.title()
        .str.replace(r"\s{2,}", " ", regex=True)
    )
    df["puntaje"]    = pd.to_numeric(df["puntaje"],  errors="coerce")
    df["merito_ep"]  = pd.to_numeric(df["merito_ep"], errors="coerce")
    df["observacion"] = df["observacion"].apply(normalizar_obs)
    df["carrera"]    = extraer_carrera(soup)
    df["url_fuente"] = url

    df = df.dropna(how="all").drop_duplicates(subset=["codigo"]).reset_index(drop=True)
    return df


def scrape_url(driver, url: str) -> pd.DataFrame:
    print(f"  Abriendo: {url}")
    html = obtener_html_completo(driver, url)
    df   = parse_tabla(html, url)
    print(f"  → {len(df)} registros extraídos.")
    return df


def guardar_csv(df: pd.DataFrame, ruta: Path):
    df.to_csv(ruta, index=False, encoding="utf-8-sig")
    print(f"\n✅ CSV guardado: {ruta.resolve()}")
    print(f"   {len(df)} filas  ×  {len(df.columns)} columnas")
    print("\n── Observaciones ──")
    print(df["observacion"].value_counts().to_string())
    if df["puntaje"].notna().any():
        print("\n── Estadísticas de puntaje ──")
        print(df["puntaje"].describe().round(3).to_string())


# ══════════════════════════════════════════════
# ▶ EJECUCIÓN
# ══════════════════════════════════════════════

driver = crear_driver()

try:
    if BATCH_RUTAS:
        dfs = []
        for ruta in BATCH_RUTAS:
            url = f"{BASE_URL}/{ruta.strip('/')}/results.html"
            df  = scrape_url(driver, url)
            if not df.empty:
                dfs.append(df)
            time.sleep(1.5)
        resultado = pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()
    else:
        resultado = scrape_url(driver, URL)
finally:
    driver.quit()
    print("[INFO] Navegador cerrado.")

if not resultado.empty:
    guardar_csv(resultado, Path(ARCHIVO_SALIDA))
    display(resultado.head(15))
else:
    print("❌ No se obtuvieron datos.")

  Abriendo: https://admision.unmsm.edu.pe/Website20262/A/036/results.html
  [✓] Paginación: forzado via DataTables JS API.
  → 52 registros extraídos.
[INFO] Navegador cerrado.

✅ CSV guardado: C:\Users\confe\OneDrive\Documentos\ANÁLISIS DE DATOS UNMSM 2026\resultados_unmsm_ARTE.csv
   52 filas  ×  8 columnas

── Observaciones ──
observacion
ALCANZÓ VACANTE          28
SIN OBSERVACIÓN          21
INHABILITADO (Art. 5)     2
AUSENTE                   1

── Estadísticas de puntaje ──
count      51.000
mean      740.547
std       156.228
min       440.500
25%       628.688
50%       745.625
75%       856.562
max      1104.750


,codigo,apellidos_nombres,escuela,puntaje,merito_ep,observacion,carrera,url_fuente
0,692707,"Alcedo Cachi, Brenda",ARTE,865.500,11.0,ALCANZÓ VACANTE,ARTE,https://admision.unmsm.edu.pe/Website20262/A/0...
1,687533,"Alva Maravi, Maria Fernanda",ARTE,1104.750,1.0,ALCANZÓ VACANTE,ARTE,https://admision.unmsm.edu.pe/Website20262/A/0...
2,688511,"Ayala Huacho, Alexia Yamile",ARTE,652.375,NaN,SIN OBSERVACIÓN,ARTE,https://admision.unmsm.edu.pe/Website20262/A/0...
3,720030,"Bendezú Urbina, Valeria Celeste",ARTE,602.875,NaN,SIN OBSERVACIÓN,ARTE,https://admision.unmsm.edu.pe/Website20262/A/0...
4,808782,"Cardenas Ninachoque, Analy Rosmery",ARTE,700.125,28.0,ALCANZÓ VACANTE,ARTE,https://admision.unmsm.edu.pe/Website20262/A/0...
5,728744,"Carrion Lozano, Anahy Valeria",ARTE,593.500,NaN,SIN OBSERVACIÓN,ARTE,https://admision.unmsm.edu.pe/Website20262/A/0...
6,721361,"Casaverde Chistama, Camila Fernanda",ARTE,789.375,20.0,ALCANZÓ VACANTE,ARTE,https://admision.unmsm.edu.pe/Website20262/A/0...
7,752532,"Chunga Diaz, Isabel Anahi",ARTE,923.125,7.0,ALCANZÓ VACANTE,ARTE,https://admision.unmsm.edu.pe/Website20262/A/0...
8,822373,"Conde Casas, Mariajose",ARTE,745.625,25.0,ALCANZÓ VACANTE,ARTE,https://admision.unmsm.edu.pe/Website20262/A/0...
9,741291,"Cuya Roman, Marcela Arianne Calipso",ARTE,632.000,NaN,SIN OBSERVACIÓN,ARTE,https://admision.unmsm.edu.pe/Website20262/A/0...


In [38]:
import pandas as pd

df = pd.read_csv("resultados_unmsm_ARTE.csv")

vacantes = df[df["observacion"] == "ALCANZÓ VACANTE"]

max_p = vacantes["puntaje"].max()
min_p = vacantes["puntaje"].min()

print(f"Carrera: {df['carrera'].iloc[0]}")
print(f"Vacantes obtenidas: {len(vacantes)}")
print(f"Puntaje máximo: {max_p}")
print(f"Puntaje mínimo: {min_p}")

display(vacantes[["codigo","apellidos_nombres","puntaje","merito_ep"]]
        .sort_values("puntaje", ascending=False)
        .reset_index(drop=True))

Carrera: ARTE
Vacantes obtenidas: 28
Puntaje máximo: 1104.75
Puntaje mínimo: 700.125


,codigo,apellidos_nombres,puntaje,merito_ep
0,687533,"Alva Maravi, Maria Fernanda",1104.750,1.0
1,769337,"De La Cruz Vidal, Treisy",1005.375,2.0
2,687839,"Huatuco Jimenez, Qary Gabriel",1002.625,3.0
3,815193,"Lopez Meza, Natalie Marisol",944.875,4.0
4,741251,"Vera Santana, Sonia Micaela",943.625,5.0
5,755250,"Santos Calvo, Danna Sharick",939.625,6.0
6,752532,"Chunga Diaz, Isabel Anahi",923.125,7.0
7,776979,"Gamboa Huatuco, Gabriel Alexander",915.750,8.0
8,753143,"Montes Cuchachi, Antonella Thays",897.500,9.0
9,762949,"Perez Verastegui, Mateo Giuliano",868.375,10.0


In [39]:
import time
import base64
import pandas as pd
from bs4 import BeautifulSoup
from pathlib import Path

from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait, Select
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager

# ══════════════════════════════════════════════
# ▶ CONFIGURACIÓN
# ══════════════════════════════════════════════

URL            = "https://admision.unmsm.edu.pe/Website20262/A/113/results.html"
BATCH_RUTAS    = []   # ejemplo: ["A/091", "A/092"]
ARCHIVO_SALIDA = "resultados_unmsm_auditoria.csv"
BASE_URL       = "https://admision.unmsm.edu.pe/Website20262"
WAIT_SEGUNDOS  = 8

# ══════════════════════════════════════════════


def crear_driver():
    opciones = Options()
    opciones.add_argument("--headless=new")
    opciones.add_argument("--no-sandbox")
    opciones.add_argument("--disable-dev-shm-usage")
    opciones.add_argument("--disable-gpu")
    opciones.add_argument("--window-size=1920,1080")
    opciones.add_argument(
        "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36"
    )
    return webdriver.Chrome(
        service=Service(ChromeDriverManager().install()),
        options=opciones
    )


def mostrar_todas_las_filas(driver):
    """
    Intenta mostrar TODOS los registros de una vez usando 3 estrategias:
    1. Cambiar el select de longitud de DataTables a -1 (All) vía Selenium
    2. Forzarlo por JavaScript directamente sobre la instancia DataTable
    3. Si no hay selector, navegar página por página
    """
    wait = WebDriverWait(driver, 10)

    # ── Estrategia 1: select de longitud (típico en DataTables) ──────────
    try:
        select_el = wait.until(
            EC.presence_of_element_located((By.CSS_SELECTOR, "select[name*='length'], select[name*='DataTables']"))
        )
        select = Select(select_el)
        try:
            select.select_by_value("-1")
            print("  [✓] Paginación: seleccionado 'Todos' en el select.")
        except Exception:
            opciones_vals = [o.get_attribute("value") for o in select.options]
            select.select_by_value(opciones_vals[-1])
            print(f"  [✓] Paginación: seleccionado valor máximo ({opciones_vals[-1]}) en el select.")
        time.sleep(3)
        return "select"
    except Exception:
        pass

    # ── Estrategia 2: forzar via JavaScript DataTables API ───────────────
    try:
        driver.execute_script("""
            var tables = $.fn.dataTable ? $.fn.dataTable.tables() : [];
            if (tables.length > 0) {
                $(tables[0]).DataTable().page.len(-1).draw();
            }
        """)
        time.sleep(3)
        print("  [✓] Paginación: forzado via DataTables JS API.")
        return "js"
    except Exception:
        pass

    # ── Estrategia 3: paginación manual (clic en "Siguiente") ────────────
    print("  [!] Usando paginación manual página por página...")
    return "manual"


def obtener_html_completo(driver, url: str) -> str:
    """Carga la página y retorna el HTML con TODOS los registros visibles."""
    driver.get(url)
    time.sleep(WAIT_SEGUNDOS)

    modo = mostrar_todas_las_filas(driver)

    if modo == "manual":
        return obtener_html_paginado(driver)

    return driver.page_source


def obtener_html_paginado(driver) -> str:
    """
    Recorre todas las páginas haciendo clic en 'Siguiente'
    y acumula los <tr> de cada página en una tabla unificada.
    """
    todas_filas_html = []
    pagina = 1

    while True:
        soup  = BeautifulSoup(driver.page_source, "html.parser")
        tabla = soup.find("table")
        if tabla:
            filas = tabla.find_all("tr")[1:]  # sin encabezado
            todas_filas_html.extend([str(f) for f in filas])
            print(f"    Página {pagina}: {len(filas)} filas | total acumulado: {len(todas_filas_html)}")

        try:
            siguiente = driver.find_element(
                By.CSS_SELECTOR,
                "a.paginate_button.next:not(.disabled), button.paginate_button.next:not(.disabled)"
            )
            siguiente.click()
            pagina += 1
            time.sleep(2)
        except Exception:
            print(f"    Fin de paginación en página {pagina}.")
            break

    # Reconstruir HTML con todas las filas
    soup_base  = BeautifulSoup(driver.page_source, "html.parser")
    tabla_base = soup_base.find("table")
    if tabla_base:
        tbody = tabla_base.find("tbody")
        if tbody:
            tbody.clear()
            for fila_html in todas_filas_html:
                tbody.append(BeautifulSoup(fila_html, "html.parser"))
    return str(soup_base)


def decodificar_b64(valor: str) -> str:
    try:
        return base64.b64decode(valor).decode("utf-8").strip()
    except Exception:
        return valor.strip()


def extraer_carrera(soup: BeautifulSoup) -> str:
    items = soup.select("ol li, ul.breadcrumb li")
    if items:
        return items[-1].get_text(strip=True)
    h1 = soup.find("h1")
    return h1.get_text(strip=True) if h1 else "DESCONOCIDA"


def normalizar_obs(val: str) -> str:
    v = val.strip().upper()
    if "ALCANZ" in v and "VACANTE" in v:
        return "ALCANZÓ VACANTE"
    if "AUSENTE" in v:
        return "AUSENTE"
    if "ART" in v:
        return "INHABILITADO (Art. 5)"
    return v if v else "SIN OBSERVACIÓN"


def parse_tabla(html: str, url: str) -> pd.DataFrame:
    soup  = BeautifulSoup(html, "html.parser")
    tabla = soup.find("table")

    if tabla is None:
        print(f"[AVISO] No se encontró tabla en: {url}")
        return pd.DataFrame()

    filas = []
    for tr in tabla.find_all("tr")[1:]:
        celdas = tr.find_all("td")
        if not celdas:
            continue

        codigo  = celdas[0].get_text(strip=True) if len(celdas) > 0 else ""

        nombre = ""
        if len(celdas) > 1:
            span = celdas[1].find("span", class_="obfuscated")
            nombre = decodificar_b64(span["data-auth"]) if (span and span.get("data-auth")) else celdas[1].get_text(strip=True)

        escuela = ""
        if len(celdas) > 2:
            span = celdas[2].find("span", class_="obfuscated")
            escuela = decodificar_b64(span["data-auth"]) if (span and span.get("data-auth")) else celdas[2].get_text(strip=True)

        puntaje = celdas[3].get("data-score", "").strip() if len(celdas) > 3 else ""
        merito  = celdas[4].get("data-merit", "").strip() if len(celdas) > 4 else ""
        obs     = celdas[5].get_text(strip=True)          if len(celdas) > 5 else ""

        filas.append({
            "codigo":            codigo,
            "apellidos_nombres": nombre,
            "escuela":           escuela,
            "puntaje":           puntaje,
            "merito_ep":         merito,
            "observacion":       obs,
        })

    if not filas:
        print(f"[AVISO] Sin filas en: {url}")
        return pd.DataFrame()

    df = pd.DataFrame(filas)
    df["codigo"] = df["codigo"].str.strip()
    df["apellidos_nombres"] = (
        df["apellidos_nombres"].str.strip().str.title()
        .str.replace(r"\s{2,}", " ", regex=True)
    )
    df["puntaje"]    = pd.to_numeric(df["puntaje"],  errors="coerce")
    df["merito_ep"]  = pd.to_numeric(df["merito_ep"], errors="coerce")
    df["observacion"] = df["observacion"].apply(normalizar_obs)
    df["carrera"]    = extraer_carrera(soup)
    df["url_fuente"] = url

    df = df.dropna(how="all").drop_duplicates(subset=["codigo"]).reset_index(drop=True)
    return df


def scrape_url(driver, url: str) -> pd.DataFrame:
    print(f"  Abriendo: {url}")
    html = obtener_html_completo(driver, url)
    df   = parse_tabla(html, url)
    print(f"  → {len(df)} registros extraídos.")
    return df


def guardar_csv(df: pd.DataFrame, ruta: Path):
    df.to_csv(ruta, index=False, encoding="utf-8-sig")
    print(f"\n✅ CSV guardado: {ruta.resolve()}")
    print(f"   {len(df)} filas  ×  {len(df.columns)} columnas")
    print("\n── Observaciones ──")
    print(df["observacion"].value_counts().to_string())
    if df["puntaje"].notna().any():
        print("\n── Estadísticas de puntaje ──")
        print(df["puntaje"].describe().round(3).to_string())


# ══════════════════════════════════════════════
# ▶ EJECUCIÓN
# ══════════════════════════════════════════════

driver = crear_driver()

try:
    if BATCH_RUTAS:
        dfs = []
        for ruta in BATCH_RUTAS:
            url = f"{BASE_URL}/{ruta.strip('/')}/results.html"
            df  = scrape_url(driver, url)
            if not df.empty:
                dfs.append(df)
            time.sleep(1.5)
        resultado = pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()
    else:
        resultado = scrape_url(driver, URL)
finally:
    driver.quit()
    print("[INFO] Navegador cerrado.")

if not resultado.empty:
    guardar_csv(resultado, Path(ARCHIVO_SALIDA))
    display(resultado.head(15))
else:
    print("❌ No se obtuvieron datos.")

  Abriendo: https://admision.unmsm.edu.pe/Website20262/A/113/results.html
  [✓] Paginación: forzado via DataTables JS API.
  → 121 registros extraídos.
[INFO] Navegador cerrado.

✅ CSV guardado: C:\Users\confe\OneDrive\Documentos\ANÁLISIS DE DATOS UNMSM 2026\resultados_unmsm_auditoria.csv
   121 filas  ×  8 columnas

── Observaciones ──
observacion
ALCANZÓ VACANTE    69
SIN OBSERVACIÓN    52

── Estadísticas de puntaje ──
count     121.000
mean      834.625
std       154.739
min       456.750
25%       725.000
50%       846.625
75%       936.250
max      1320.000


,codigo,apellidos_nombres,escuela,puntaje,merito_ep,observacion,carrera,url_fuente
0,659315,"Alarcon Infante, Anthony",AUDITORÍA EMPRESARIAL Y DEL SECTOR PÚBLICO,967.875,23.0,ALCANZÓ VACANTE,AUDITORÍA EMPRESARIAL Y DEL SECTOR PÚBLICO,https://admision.unmsm.edu.pe/Website20262/A/1...
1,656040,"Aliaga Perez, Ariana Nicol",AUDITORÍA EMPRESARIAL Y DEL SECTOR PÚBLICO,822.000,NaN,SIN OBSERVACIÓN,AUDITORÍA EMPRESARIAL Y DEL SECTOR PÚBLICO,https://admision.unmsm.edu.pe/Website20262/A/1...
2,589570,"Allende Lopez, Rosse Nayely",AUDITORÍA EMPRESARIAL Y DEL SECTOR PÚBLICO,837.250,64.0,ALCANZÓ VACANTE,AUDITORÍA EMPRESARIAL Y DEL SECTOR PÚBLICO,https://admision.unmsm.edu.pe/Website20262/A/1...
3,594816,"Apolinario Ramos, Kiara",AUDITORÍA EMPRESARIAL Y DEL SECTOR PÚBLICO,936.250,31.0,ALCANZÓ VACANTE,AUDITORÍA EMPRESARIAL Y DEL SECTOR PÚBLICO,https://admision.unmsm.edu.pe/Website20262/A/1...
4,653278,"Arias Velasque, Cesar Alonzo",AUDITORÍA EMPRESARIAL Y DEL SECTOR PÚBLICO,879.750,46.0,ALCANZÓ VACANTE,AUDITORÍA EMPRESARIAL Y DEL SECTOR PÚBLICO,https://admision.unmsm.edu.pe/Website20262/A/1...
5,651401,"Arzapalo Poma, Jacquelin Melina",AUDITORÍA EMPRESARIAL Y DEL SECTOR PÚBLICO,661.125,NaN,SIN OBSERVACIÓN,AUDITORÍA EMPRESARIAL Y DEL SECTOR PÚBLICO,https://admision.unmsm.edu.pe/Website20262/A/1...
6,650065,"Aviles Pozo, Ithmer Edwin",AUDITORÍA EMPRESARIAL Y DEL SECTOR PÚBLICO,554.750,NaN,SIN OBSERVACIÓN,AUDITORÍA EMPRESARIAL Y DEL SECTOR PÚBLICO,https://admision.unmsm.edu.pe/Website20262/A/1...
7,652955,"Ayquipa Pasión, Valeria Jazmín",AUDITORÍA EMPRESARIAL Y DEL SECTOR PÚBLICO,1038.500,12.0,ALCANZÓ VACANTE,AUDITORÍA EMPRESARIAL Y DEL SECTOR PÚBLICO,https://admision.unmsm.edu.pe/Website20262/A/1...
8,571241,"Bautista Medina, Xiomara Ximena",AUDITORÍA EMPRESARIAL Y DEL SECTOR PÚBLICO,858.625,56.0,ALCANZÓ VACANTE,AUDITORÍA EMPRESARIAL Y DEL SECTOR PÚBLICO,https://admision.unmsm.edu.pe/Website20262/A/1...
9,645799,"Briceño Mallaupoma, Jhorland Sebastian",AUDITORÍA EMPRESARIAL Y DEL SECTOR PÚBLICO,620.000,NaN,SIN OBSERVACIÓN,AUDITORÍA EMPRESARIAL Y DEL SECTOR PÚBLICO,https://admision.unmsm.edu.pe/Website20262/A/1...


In [40]:
import pandas as pd

df = pd.read_csv("resultados_unmsm_auditoria.csv")

vacantes = df[df["observacion"] == "ALCANZÓ VACANTE"]

max_p = vacantes["puntaje"].max()
min_p = vacantes["puntaje"].min()

print(f"Carrera: {df['carrera'].iloc[0]}")
print(f"Vacantes obtenidas: {len(vacantes)}")
print(f"Puntaje máximo: {max_p}")
print(f"Puntaje mínimo: {min_p}")

display(vacantes[["codigo","apellidos_nombres","puntaje","merito_ep"]]
        .sort_values("puntaje", ascending=False)
        .reset_index(drop=True))

Carrera: AUDITORÍA EMPRESARIAL Y DEL SECTOR PÚBLICO
Vacantes obtenidas: 69
Puntaje máximo: 1320.0
Puntaje mínimo: 826.125


,codigo,apellidos_nombres,puntaje,merito_ep
0,573671,"De La Cruz Cerna, Max Alfredo",1320.000,1.0
1,555831,"Huiza Gomez, Gianfranco Tiago",1177.250,2.0
2,570653,"Carrillo Huacho, Carmen Cielo",1145.000,3.0
3,562203,"Viera Clemente, Isabela",1112.625,4.0
4,646132,"Tejeda Espinoza, Keith Nicole",1101.875,5.0
...,...,...,...,...
64,650141,"Pintado Obregon, Adriano Alexis",835.250,65.0
65,655811,"Reategui Castillo, Frank Alexander",835.250,66.0
66,656501,"Paredes Camacho, Fabio Andre",832.875,67.0
67,655257,"Monteverde Alarcon, Lya Fernanda",826.750,68.0


In [41]:
import time
import base64
import pandas as pd
from bs4 import BeautifulSoup
from pathlib import Path

from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait, Select
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager

# ══════════════════════════════════════════════
# ▶ CONFIGURACIÓN
# ══════════════════════════════════════════════

URL            = "https://admision.unmsm.edu.pe/Website20262/A/037/results.html"
BATCH_RUTAS    = []   # ejemplo: ["A/091", "A/092"]
ARCHIVO_SALIDA = "resultados_unmsm_biblio.csv"
BASE_URL       = "https://admision.unmsm.edu.pe/Website20262"
WAIT_SEGUNDOS  = 8

# ══════════════════════════════════════════════


def crear_driver():
    opciones = Options()
    opciones.add_argument("--headless=new")
    opciones.add_argument("--no-sandbox")
    opciones.add_argument("--disable-dev-shm-usage")
    opciones.add_argument("--disable-gpu")
    opciones.add_argument("--window-size=1920,1080")
    opciones.add_argument(
        "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36"
    )
    return webdriver.Chrome(
        service=Service(ChromeDriverManager().install()),
        options=opciones
    )


def mostrar_todas_las_filas(driver):
    """
    Intenta mostrar TODOS los registros de una vez usando 3 estrategias:
    1. Cambiar el select de longitud de DataTables a -1 (All) vía Selenium
    2. Forzarlo por JavaScript directamente sobre la instancia DataTable
    3. Si no hay selector, navegar página por página
    """
    wait = WebDriverWait(driver, 10)

    # ── Estrategia 1: select de longitud (típico en DataTables) ──────────
    try:
        select_el = wait.until(
            EC.presence_of_element_located((By.CSS_SELECTOR, "select[name*='length'], select[name*='DataTables']"))
        )
        select = Select(select_el)
        try:
            select.select_by_value("-1")
            print("  [✓] Paginación: seleccionado 'Todos' en el select.")
        except Exception:
            opciones_vals = [o.get_attribute("value") for o in select.options]
            select.select_by_value(opciones_vals[-1])
            print(f"  [✓] Paginación: seleccionado valor máximo ({opciones_vals[-1]}) en el select.")
        time.sleep(3)
        return "select"
    except Exception:
        pass

    # ── Estrategia 2: forzar via JavaScript DataTables API ───────────────
    try:
        driver.execute_script("""
            var tables = $.fn.dataTable ? $.fn.dataTable.tables() : [];
            if (tables.length > 0) {
                $(tables[0]).DataTable().page.len(-1).draw();
            }
        """)
        time.sleep(3)
        print("  [✓] Paginación: forzado via DataTables JS API.")
        return "js"
    except Exception:
        pass

    # ── Estrategia 3: paginación manual (clic en "Siguiente") ────────────
    print("  [!] Usando paginación manual página por página...")
    return "manual"


def obtener_html_completo(driver, url: str) -> str:
    """Carga la página y retorna el HTML con TODOS los registros visibles."""
    driver.get(url)
    time.sleep(WAIT_SEGUNDOS)

    modo = mostrar_todas_las_filas(driver)

    if modo == "manual":
        return obtener_html_paginado(driver)

    return driver.page_source


def obtener_html_paginado(driver) -> str:
    """
    Recorre todas las páginas haciendo clic en 'Siguiente'
    y acumula los <tr> de cada página en una tabla unificada.
    """
    todas_filas_html = []
    pagina = 1

    while True:
        soup  = BeautifulSoup(driver.page_source, "html.parser")
        tabla = soup.find("table")
        if tabla:
            filas = tabla.find_all("tr")[1:]  # sin encabezado
            todas_filas_html.extend([str(f) for f in filas])
            print(f"    Página {pagina}: {len(filas)} filas | total acumulado: {len(todas_filas_html)}")

        try:
            siguiente = driver.find_element(
                By.CSS_SELECTOR,
                "a.paginate_button.next:not(.disabled), button.paginate_button.next:not(.disabled)"
            )
            siguiente.click()
            pagina += 1
            time.sleep(2)
        except Exception:
            print(f"    Fin de paginación en página {pagina}.")
            break

    # Reconstruir HTML con todas las filas
    soup_base  = BeautifulSoup(driver.page_source, "html.parser")
    tabla_base = soup_base.find("table")
    if tabla_base:
        tbody = tabla_base.find("tbody")
        if tbody:
            tbody.clear()
            for fila_html in todas_filas_html:
                tbody.append(BeautifulSoup(fila_html, "html.parser"))
    return str(soup_base)


def decodificar_b64(valor: str) -> str:
    try:
        return base64.b64decode(valor).decode("utf-8").strip()
    except Exception:
        return valor.strip()


def extraer_carrera(soup: BeautifulSoup) -> str:
    items = soup.select("ol li, ul.breadcrumb li")
    if items:
        return items[-1].get_text(strip=True)
    h1 = soup.find("h1")
    return h1.get_text(strip=True) if h1 else "DESCONOCIDA"


def normalizar_obs(val: str) -> str:
    v = val.strip().upper()
    if "ALCANZ" in v and "VACANTE" in v:
        return "ALCANZÓ VACANTE"
    if "AUSENTE" in v:
        return "AUSENTE"
    if "ART" in v:
        return "INHABILITADO (Art. 5)"
    return v if v else "SIN OBSERVACIÓN"


def parse_tabla(html: str, url: str) -> pd.DataFrame:
    soup  = BeautifulSoup(html, "html.parser")
    tabla = soup.find("table")

    if tabla is None:
        print(f"[AVISO] No se encontró tabla en: {url}")
        return pd.DataFrame()

    filas = []
    for tr in tabla.find_all("tr")[1:]:
        celdas = tr.find_all("td")
        if not celdas:
            continue

        codigo  = celdas[0].get_text(strip=True) if len(celdas) > 0 else ""

        nombre = ""
        if len(celdas) > 1:
            span = celdas[1].find("span", class_="obfuscated")
            nombre = decodificar_b64(span["data-auth"]) if (span and span.get("data-auth")) else celdas[1].get_text(strip=True)

        escuela = ""
        if len(celdas) > 2:
            span = celdas[2].find("span", class_="obfuscated")
            escuela = decodificar_b64(span["data-auth"]) if (span and span.get("data-auth")) else celdas[2].get_text(strip=True)

        puntaje = celdas[3].get("data-score", "").strip() if len(celdas) > 3 else ""
        merito  = celdas[4].get("data-merit", "").strip() if len(celdas) > 4 else ""
        obs     = celdas[5].get_text(strip=True)          if len(celdas) > 5 else ""

        filas.append({
            "codigo":            codigo,
            "apellidos_nombres": nombre,
            "escuela":           escuela,
            "puntaje":           puntaje,
            "merito_ep":         merito,
            "observacion":       obs,
        })

    if not filas:
        print(f"[AVISO] Sin filas en: {url}")
        return pd.DataFrame()

    df = pd.DataFrame(filas)
    df["codigo"] = df["codigo"].str.strip()
    df["apellidos_nombres"] = (
        df["apellidos_nombres"].str.strip().str.title()
        .str.replace(r"\s{2,}", " ", regex=True)
    )
    df["puntaje"]    = pd.to_numeric(df["puntaje"],  errors="coerce")
    df["merito_ep"]  = pd.to_numeric(df["merito_ep"], errors="coerce")
    df["observacion"] = df["observacion"].apply(normalizar_obs)
    df["carrera"]    = extraer_carrera(soup)
    df["url_fuente"] = url

    df = df.dropna(how="all").drop_duplicates(subset=["codigo"]).reset_index(drop=True)
    return df


def scrape_url(driver, url: str) -> pd.DataFrame:
    print(f"  Abriendo: {url}")
    html = obtener_html_completo(driver, url)
    df   = parse_tabla(html, url)
    print(f"  → {len(df)} registros extraídos.")
    return df


def guardar_csv(df: pd.DataFrame, ruta: Path):
    df.to_csv(ruta, index=False, encoding="utf-8-sig")
    print(f"\n✅ CSV guardado: {ruta.resolve()}")
    print(f"   {len(df)} filas  ×  {len(df.columns)} columnas")
    print("\n── Observaciones ──")
    print(df["observacion"].value_counts().to_string())
    if df["puntaje"].notna().any():
        print("\n── Estadísticas de puntaje ──")
        print(df["puntaje"].describe().round(3).to_string())


# ══════════════════════════════════════════════
# ▶ EJECUCIÓN
# ══════════════════════════════════════════════

driver = crear_driver()

try:
    if BATCH_RUTAS:
        dfs = []
        for ruta in BATCH_RUTAS:
            url = f"{BASE_URL}/{ruta.strip('/')}/results.html"
            df  = scrape_url(driver, url)
            if not df.empty:
                dfs.append(df)
            time.sleep(1.5)
        resultado = pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()
    else:
        resultado = scrape_url(driver, URL)
finally:
    driver.quit()
    print("[INFO] Navegador cerrado.")

if not resultado.empty:
    guardar_csv(resultado, Path(ARCHIVO_SALIDA))
    display(resultado.head(15))
else:
    print("❌ No se obtuvieron datos.")

  Abriendo: https://admision.unmsm.edu.pe/Website20262/A/037/results.html
  [✓] Paginación: forzado via DataTables JS API.
  → 83 registros extraídos.
[INFO] Navegador cerrado.

✅ CSV guardado: C:\Users\confe\OneDrive\Documentos\ANÁLISIS DE DATOS UNMSM 2026\resultados_unmsm_biblio.csv
   83 filas  ×  8 columnas

── Observaciones ──
observacion
SIN OBSERVACIÓN    55
ALCANZÓ VACANTE    27
AUSENTE             1

── Estadísticas de puntaje ──
count      82.000
mean      794.363
std       144.651
min       384.375
25%       706.719
50%       788.188
75%       892.062
max      1143.500


,codigo,apellidos_nombres,escuela,puntaje,merito_ep,observacion,carrera,url_fuente
0,835879,"Abanto Muñoz, Jimena Guadalupe",BIBLIOTECOLOGÍA Y CIENCIAS DE LA INFORMACIÓN,793.125,NaN,SIN OBSERVACIÓN,BIBLIOTECOLOGÍA Y CIENCIAS DE LA INFORMACIÓN,https://admision.unmsm.edu.pe/Website20262/A/0...
1,776252,"Acevedo Neciosup, Evelyn Johana",BIBLIOTECOLOGÍA Y CIENCIAS DE LA INFORMACIÓN,708.500,NaN,SIN OBSERVACIÓN,BIBLIOTECOLOGÍA Y CIENCIAS DE LA INFORMACIÓN,https://admision.unmsm.edu.pe/Website20262/A/0...
2,783177,"Ampuero Muñiz, Franchesca Delia",BIBLIOTECOLOGÍA Y CIENCIAS DE LA INFORMACIÓN,855.250,24.0,ALCANZÓ VACANTE,BIBLIOTECOLOGÍA Y CIENCIAS DE LA INFORMACIÓN,https://admision.unmsm.edu.pe/Website20262/A/0...
3,806028,"Aranibar Chucchu, Leydi Vanessa",BIBLIOTECOLOGÍA Y CIENCIAS DE LA INFORMACIÓN,732.875,NaN,SIN OBSERVACIÓN,BIBLIOTECOLOGÍA Y CIENCIAS DE LA INFORMACIÓN,https://admision.unmsm.edu.pe/Website20262/A/0...
4,793594,"Arzapalo Inga, Keith Charles",BIBLIOTECOLOGÍA Y CIENCIAS DE LA INFORMACIÓN,767.250,NaN,SIN OBSERVACIÓN,BIBLIOTECOLOGÍA Y CIENCIAS DE LA INFORMACIÓN,https://admision.unmsm.edu.pe/Website20262/A/0...
5,726298,"Atauqui Rios, Victoria Alexandra",BIBLIOTECOLOGÍA Y CIENCIAS DE LA INFORMACIÓN,736.625,NaN,SIN OBSERVACIÓN,BIBLIOTECOLOGÍA Y CIENCIAS DE LA INFORMACIÓN,https://admision.unmsm.edu.pe/Website20262/A/0...
6,797735,"Atoche Linares, Ronald David",BIBLIOTECOLOGÍA Y CIENCIAS DE LA INFORMACIÓN,988.375,9.0,ALCANZÓ VACANTE,BIBLIOTECOLOGÍA Y CIENCIAS DE LA INFORMACIÓN,https://admision.unmsm.edu.pe/Website20262/A/0...
7,769034,"Ayala Segama, Marina Nicole",BIBLIOTECOLOGÍA Y CIENCIAS DE LA INFORMACIÓN,600.875,NaN,SIN OBSERVACIÓN,BIBLIOTECOLOGÍA Y CIENCIAS DE LA INFORMACIÓN,https://admision.unmsm.edu.pe/Website20262/A/0...
8,754840,"Bardales Navarro, Jorge Elmer",BIBLIOTECOLOGÍA Y CIENCIAS DE LA INFORMACIÓN,830.125,NaN,SIN OBSERVACIÓN,BIBLIOTECOLOGÍA Y CIENCIAS DE LA INFORMACIÓN,https://admision.unmsm.edu.pe/Website20262/A/0...
9,687689,"Calderon Canahuire, Jorge Armando",BIBLIOTECOLOGÍA Y CIENCIAS DE LA INFORMACIÓN,812.875,NaN,SIN OBSERVACIÓN,BIBLIOTECOLOGÍA Y CIENCIAS DE LA INFORMACIÓN,https://admision.unmsm.edu.pe/Website20262/A/0...


In [42]:
import pandas as pd

df = pd.read_csv("resultados_unmsm_biblio.csv")

vacantes = df[df["observacion"] == "ALCANZÓ VACANTE"]

max_p = vacantes["puntaje"].max()
min_p = vacantes["puntaje"].min()

print(f"Carrera: {df['carrera'].iloc[0]}")
print(f"Vacantes obtenidas: {len(vacantes)}")
print(f"Puntaje máximo: {max_p}")
print(f"Puntaje mínimo: {min_p}")

display(vacantes[["codigo","apellidos_nombres","puntaje","merito_ep"]]
        .sort_values("puntaje", ascending=False)
        .reset_index(drop=True))

Carrera: BIBLIOTECOLOGÍA Y CIENCIAS DE LA INFORMACIÓN
Vacantes obtenidas: 27
Puntaje máximo: 1143.5
Puntaje mínimo: 841.5


,codigo,apellidos_nombres,puntaje,merito_ep
0,833414,"Rivas Mendoza, Milagros Nicolt",1143.500,1.0
1,686872,"Yglesias Cruz, Marcos Daniel",1089.875,2.0
2,832419,"Santos Saucedo, Adrian Alberto",1084.750,3.0
3,753942,"Leandro Ruiz, Melina Kimberly",1071.625,4.0
4,722044,"Salazar Vasquez, Roberth Francisco",1067.625,5.0
5,773222,"Peña Chuquihuanga, Ayde Alicia",1040.750,6.0
6,712846,"Villar Soto, Maria Magdalena",1011.875,7.0
7,754020,"Hilario Condor, Mark Brian",991.625,8.0
8,797735,"Atoche Linares, Ronald David",988.375,9.0
9,834544,"Miranda Huaytalla, Jasmin Tatiana",963.875,10.0


In [ ]:
import time
import base64
import pandas as pd
from bs4 import BeautifulSoup
from pathlib import Path

from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait, Select
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager

# ══════════════════════════════════════════════
# ▶ CONFIGURACIÓN
# ══════════════════════════════════════════════

URL            = "https://admision.unmsm.edu.pe/Website20262/A/203/results.html"
BATCH_RUTAS    = []   # ejemplo: ["A/091", "A/092"]
ARCHIVO_SALIDA = "resultados_unmsm_computacion.csv"
BASE_URL       = "https://admision.unmsm.edu.pe/Website20262"
WAIT_SEGUNDOS  = 8

# ══════════════════════════════════════════════


def crear_driver():
    opciones = Options()
    opciones.add_argument("--headless=new")
    opciones.add_argument("--no-sandbox")
    opciones.add_argument("--disable-dev-shm-usage")
    opciones.add_argument("--disable-gpu")
    opciones.add_argument("--window-size=1920,1080")
    opciones.add_argument(
        "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36"
    )
    return webdriver.Chrome(
        service=Service(ChromeDriverManager().install()),
        options=opciones
    )


def mostrar_todas_las_filas(driver):
    """
    Intenta mostrar TODOS los registros de una vez usando 3 estrategias:
    1. Cambiar el select de longitud de DataTables a -1 (All) vía Selenium
    2. Forzarlo por JavaScript directamente sobre la instancia DataTable
    3. Si no hay selector, navegar página por página
    """
    wait = WebDriverWait(driver, 10)

    # ── Estrategia 1: select de longitud (típico en DataTables) ──────────
    try:
        select_el = wait.until(
            EC.presence_of_element_located((By.CSS_SELECTOR, "select[name*='length'], select[name*='DataTables']"))
        )
        select = Select(select_el)
        try:
            select.select_by_value("-1")
            print("  [✓] Paginación: seleccionado 'Todos' en el select.")
        except Exception:
            opciones_vals = [o.get_attribute("value") for o in select.options]
            select.select_by_value(opciones_vals[-1])
            print(f"  [✓] Paginación: seleccionado valor máximo ({opciones_vals[-1]}) en el select.")
        time.sleep(3)
        return "select"
    except Exception:
        pass

    # ── Estrategia 2: forzar via JavaScript DataTables API ───────────────
    try:
        driver.execute_script("""
            var tables = $.fn.dataTable ? $.fn.dataTable.tables() : [];
            if (tables.length > 0) {
                $(tables[0]).DataTable().page.len(-1).draw();
            }
        """)
        time.sleep(3)
        print("  [✓] Paginación: forzado via DataTables JS API.")
        return "js"
    except Exception:
        pass

    # ── Estrategia 3: paginación manual (clic en "Siguiente") ────────────
    print("  [!] Usando paginación manual página por página...")
    return "manual"


def obtener_html_completo(driver, url: str) -> str:
    """Carga la página y retorna el HTML con TODOS los registros visibles."""
    driver.get(url)
    time.sleep(WAIT_SEGUNDOS)

    modo = mostrar_todas_las_filas(driver)

    if modo == "manual":
        return obtener_html_paginado(driver)

    return driver.page_source


def obtener_html_paginado(driver) -> str:
    """
    Recorre todas las páginas haciendo clic en 'Siguiente'
    y acumula los <tr> de cada página en una tabla unificada.
    """
    todas_filas_html = []
    pagina = 1

    while True:
        soup  = BeautifulSoup(driver.page_source, "html.parser")
        tabla = soup.find("table")
        if tabla:
            filas = tabla.find_all("tr")[1:]  # sin encabezado
            todas_filas_html.extend([str(f) for f in filas])
            print(f"    Página {pagina}: {len(filas)} filas | total acumulado: {len(todas_filas_html)}")

        try:
            siguiente = driver.find_element(
                By.CSS_SELECTOR,
                "a.paginate_button.next:not(.disabled), button.paginate_button.next:not(.disabled)"
            )
            siguiente.click()
            pagina += 1
            time.sleep(2)
        except Exception:
            print(f"    Fin de paginación en página {pagina}.")
            break

    # Reconstruir HTML con todas las filas
    soup_base  = BeautifulSoup(driver.page_source, "html.parser")
    tabla_base = soup_base.find("table")
    if tabla_base:
        tbody = tabla_base.find("tbody")
        if tbody:
            tbody.clear()
            for fila_html in todas_filas_html:
                tbody.append(BeautifulSoup(fila_html, "html.parser"))
    return str(soup_base)


def decodificar_b64(valor: str) -> str:
    try:
        return base64.b64decode(valor).decode("utf-8").strip()
    except Exception:
        return valor.strip()


def extraer_carrera(soup: BeautifulSoup) -> str:
    items = soup.select("ol li, ul.breadcrumb li")
    if items:
        return items[-1].get_text(strip=True)
    h1 = soup.find("h1")
    return h1.get_text(strip=True) if h1 else "DESCONOCIDA"


def normalizar_obs(val: str) -> str:
    v = val.strip().upper()
    if "ALCANZ" in v and "VACANTE" in v:
        return "ALCANZÓ VACANTE"
    if "AUSENTE" in v:
        return "AUSENTE"
    if "ART" in v:
        return "INHABILITADO (Art. 5)"
    return v if v else "SIN OBSERVACIÓN"


def parse_tabla(html: str, url: str) -> pd.DataFrame:
    soup  = BeautifulSoup(html, "html.parser")
    tabla = soup.find("table")

    if tabla is None:
        print(f"[AVISO] No se encontró tabla en: {url}")
        return pd.DataFrame()

    filas = []
    for tr in tabla.find_all("tr")[1:]:
        celdas = tr.find_all("td")
        if not celdas:
            continue

        codigo  = celdas[0].get_text(strip=True) if len(celdas) > 0 else ""

        nombre = ""
        if len(celdas) > 1:
            span = celdas[1].find("span", class_="obfuscated")
            nombre = decodificar_b64(span["data-auth"]) if (span and span.get("data-auth")) else celdas[1].get_text(strip=True)

        escuela = ""
        if len(celdas) > 2:
            span = celdas[2].find("span", class_="obfuscated")
            escuela = decodificar_b64(span["data-auth"]) if (span and span.get("data-auth")) else celdas[2].get_text(strip=True)

        puntaje = celdas[3].get("data-score", "").strip() if len(celdas) > 3 else ""
        merito  = celdas[4].get("data-merit", "").strip() if len(celdas) > 4 else ""
        obs     = celdas[5].get_text(strip=True)          if len(celdas) > 5 else ""

        filas.append({
            "codigo":            codigo,
            "apellidos_nombres": nombre,
            "escuela":           escuela,
            "puntaje":           puntaje,
            "merito_ep":         merito,
            "observacion":       obs,
        })

    if not filas:
        print(f"[AVISO] Sin filas en: {url}")
        return pd.DataFrame()

    df = pd.DataFrame(filas)
    df["codigo"] = df["codigo"].str.strip()
    df["apellidos_nombres"] = (
        df["apellidos_nombres"].str.strip().str.title()
        .str.replace(r"\s{2,}", " ", regex=True)
    )
    df["puntaje"]    = pd.to_numeric(df["puntaje"],  errors="coerce")
    df["merito_ep"]  = pd.to_numeric(df["merito_ep"], errors="coerce")
    df["observacion"] = df["observacion"].apply(normalizar_obs)
    df["carrera"]    = extraer_carrera(soup)
    df["url_fuente"] = url

    df = df.dropna(how="all").drop_duplicates(subset=["codigo"]).reset_index(drop=True)
    return df


def scrape_url(driver, url: str) -> pd.DataFrame:
    print(f"  Abriendo: {url}")
    html = obtener_html_completo(driver, url)
    df   = parse_tabla(html, url)
    print(f"  → {len(df)} registros extraídos.")
    return df


def guardar_csv(df: pd.DataFrame, ruta: Path):
    df.to_csv(ruta, index=False, encoding="utf-8-sig")
    print(f"\n✅ CSV guardado: {ruta.resolve()}")
    print(f"   {len(df)} filas  ×  {len(df.columns)} columnas")
    print("\n── Observaciones ──")
    print(df["observacion"].value_counts().to_string())
    if df["puntaje"].notna().any():
        print("\n── Estadísticas de puntaje ──")
        print(df["puntaje"].describe().round(3).to_string())


# ══════════════════════════════════════════════
# ▶ EJECUCIÓN
# ══════════════════════════════════════════════

driver = crear_driver()

try:
    if BATCH_RUTAS:
        dfs = []
        for ruta in BATCH_RUTAS:
            url = f"{BASE_URL}/{ruta.strip('/')}/results.html"
            df  = scrape_url(driver, url)
            if not df.empty:
                dfs.append(df)
            time.sleep(1.5)
        resultado = pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()
    else:
        resultado = scrape_url(driver, URL)
finally:
    driver.quit()
    print("[INFO] Navegador cerrado.")

if not resultado.empty:
    guardar_csv(resultado, Path(ARCHIVO_SALIDA))
    display(resultado.head(15))
else:
    print("❌ No se obtuvieron datos.")

  Abriendo: https://admision.unmsm.edu.pe/Website20262/A/203/results.html
  [✓] Paginación: forzado via DataTables JS API.
  → 134 registros extraídos.
[INFO] Navegador cerrado.

✅ CSV guardado: C:\Users\confe\OneDrive\Documentos\ANÁLISIS DE DATOS UNMSM 2026\resultados_unmsm_computacion.csv
   134 filas  ×  8 columnas

── Observaciones ──
observacion
SIN OBSERVACIÓN          79
ALCANZÓ VACANTE          53
INHABILITADO (Art. 5)     2

── Estadísticas de puntaje ──
count     134.000
mean      973.630
std       198.307
min       509.250
25%       844.469
50%       978.500
75%      1105.594
max      1510.125


,codigo,apellidos_nombres,escuela,puntaje,merito_ep,observacion,carrera,url_fuente
0,382800,"Abarca Linares, Juvenal Yann",CIENCIA DE LA COMPUTACIÓN,940.875,NaN,SIN OBSERVACIÓN,CIENCIA DE LA COMPUTACIÓN,https://admision.unmsm.edu.pe/Website20262/A/2...
1,336783,"Abregu Coronel, Victor Andres",CIENCIA DE LA COMPUTACIÓN,1155.000,21.0,ALCANZÓ VACANTE,CIENCIA DE LA COMPUTACIÓN,https://admision.unmsm.edu.pe/Website20262/A/2...
2,379074,"Aguilar Anicama, Matías Alonso",CIENCIA DE LA COMPUTACIÓN,1204.625,15.0,ALCANZÓ VACANTE,CIENCIA DE LA COMPUTACIÓN,https://admision.unmsm.edu.pe/Website20262/A/2...
3,372924,"Alarcon Ramirez, Isabel Dennisse",CIENCIA DE LA COMPUTACIÓN,936.750,NaN,SIN OBSERVACIÓN,CIENCIA DE LA COMPUTACIÓN,https://admision.unmsm.edu.pe/Website20262/A/2...
4,367582,"Almonacid Vela, Joaquin Sebastian",CIENCIA DE LA COMPUTACIÓN,1104.750,34.0,ALCANZÓ VACANTE,CIENCIA DE LA COMPUTACIÓN,https://admision.unmsm.edu.pe/Website20262/A/2...
5,337120,"Alvarado Romero, Fher Eli",CIENCIA DE LA COMPUTACIÓN,975.125,NaN,SIN OBSERVACIÓN,CIENCIA DE LA COMPUTACIÓN,https://admision.unmsm.edu.pe/Website20262/A/2...
6,336761,"Alvarez Rosas, Rolando Benjamin",CIENCIA DE LA COMPUTACIÓN,593.625,NaN,SIN OBSERVACIÓN,CIENCIA DE LA COMPUTACIÓN,https://admision.unmsm.edu.pe/Website20262/A/2...
7,317284,"Alvarez Sivirichi, Daniel Esteban",CIENCIA DE LA COMPUTACIÓN,1004.250,NaN,SIN OBSERVACIÓN,CIENCIA DE LA COMPUTACIÓN,https://admision.unmsm.edu.pe/Website20262/A/2...
8,398423,"Alzamora Melendez, Angel Sebastian",CIENCIA DE LA COMPUTACIÓN,996.250,NaN,SIN OBSERVACIÓN,CIENCIA DE LA COMPUTACIÓN,https://admision.unmsm.edu.pe/Website20262/A/2...
9,395654,"Amaya Yupanqui, Joel Fritz",CIENCIA DE LA COMPUTACIÓN,544.125,NaN,SIN OBSERVACIÓN,CIENCIA DE LA COMPUTACIÓN,https://admision.unmsm.edu.pe/Website20262/A/2...


In [44]:
import pandas as pd

df = pd.read_csv("resultados_unmsm_computacion.csv")

vacantes = df[df["observacion"] == "ALCANZÓ VACANTE"]

max_p = vacantes["puntaje"].max()
min_p = vacantes["puntaje"].min()

print(f"Carrera: {df['carrera'].iloc[0]}")
print(f"Vacantes obtenidas: {len(vacantes)}")
print(f"Puntaje máximo: {max_p}")
print(f"Puntaje mínimo: {min_p}")

display(vacantes[["codigo","apellidos_nombres","puntaje","merito_ep"]]
        .sort_values("puntaje", ascending=False)
        .reset_index(drop=True))

Carrera: CIENCIA DE LA COMPUTACIÓN
Vacantes obtenidas: 53
Puntaje máximo: 1510.125
Puntaje mínimo: 1018.5


,codigo,apellidos_nombres,puntaje,merito_ep
0,330174,"Zevallos Cossio, Diego Aaron",1510.125,1.0
1,336091,"Quispe Gamion, Alessandra Milagros",1379.375,2.0
2,330545,"Diaz Estremadoyro, Wenzel David",1324.375,3.0
3,368760,"Gamboa Cortez, Joau Adriano",1320.500,4.0
4,369536,"Navarro Aynaya, Ismael Lee",1320.000,5.0
5,398813,"Avila Chirito, Luis Sebastian",1310.250,6.0
6,368782,"Apaza Llahuilla, Geordano Marcell",1302.875,7.0
7,370795,"Milla Condo, Frida Sofia",1273.750,8.0
8,326461,"Gutierrez Diaz, Humberto Duvan",1272.000,9.0
9,385615,"Ayón Ojeda, Jazmín",1268.000,10.0
